# MORICHIKA Final-Notebook Extended with Strict-v3 Retrieval

Fresh reference-notebook execution scaffold extended with MORICHIKA strict-v3 SQLite/NumPy retrieval and v5 contextual policy. No legacy data-dump labels or QA artifacts.


In [ ]:
from __future__ import annotations

import gc, hashlib, importlib, importlib.metadata, json, math, os, random, re
import shutil, sqlite3, stat, subprocess, sys, time, unicodedata, zipfile
from collections import Counter, defaultdict
from pathlib import Path, PurePosixPath
from typing import Any, Callable

import numpy as np
import pandas as pd

SEED = 20260720
RUN_LLM = True
MODEL_BACKEND = "q4_gguf"
MODEL_ID = "google/gemma-4-31B-it"
MODEL_PATH_OVERRIDE = ""
Q4_MODEL_ID = "google/gemma-4-31b-it-qat-q4_0-gguf"
Q4_MODEL_PATH_OVERRIDE = "/kaggle/input/models/google/gemma-4/gguf/gemma-4-31b-it-qat-q4_0-gguf/2/gemma-4-31B_q4_0-it.gguf"
Q4_N_CTX, Q4_N_BATCH, Q4_N_UBATCH = 2048, 384, 128
Q4_CONTEXT_CHAR_FALLBACKS = (6000, 3600, 2000, 1000, 400, 0)
MAX_LENGTH, BATCH_ROWS, CHECKPOINT_EVERY = 2048, 2, 25
MAX_REFERENCE_ANSWERS = 12
MAX_DELIB_TOKENS = 12
DELIB_BATCH_ROWS = 2
MAX_DELIB_SAMPLE_ROWS = MAX_DELIB_TEST_ROWS = 0
RUN_DELIBERATION = False
ALLOW_ONLINE_MODEL_FALLBACK = False
MIN_SEMANTIC_REFERENCE_SAMPLE_AGREEMENTS = 0

random.seed(SEED); np.random.seed(SEED)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)
if not INPUT_ROOT.is_dir():
    raise RuntimeError("This production notebook must run inside Kaggle")

test_path = INPUT_ROOT / "competitions/bengali-hallucination/test set.csv"
sample_path = INPUT_ROOT / "competitions/bengali-hallucination/sample submission.csv"
if not test_path.is_file() or not sample_path.is_file():
    raise FileNotFoundError("Competition test/template files are not attached")
test = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_path)
required = {"id", "context", "prompt_bn", "response_bn"}
if not required.issubset(test.columns):
    raise ValueError(f"test schema missing: {sorted(required-set(test.columns))}")
for name in ("context", "prompt_bn", "response_bn"):
    test[name] = test[name].fillna("").astype(str)
test["id"] = test["id"].astype(str)
sample_submission["id"] = sample_submission["id"].astype(str)
if test.id.duplicated().any() or test.id.tolist() != sample_submission.id.tolist():
    raise ValueError("competition id/order contract failed")

NULL_CONTEXTS = {"", "none", "null", "nan", "n/a", "na", "[null]", "[none]", "[nan]", "[n/a]", "[na]", "<null>", "<none>", "<nan>"}
def has_context(value: object) -> bool:
    folded=str(value or "").strip().casefold()
    if folded in NULL_CONTEXTS or re.fullmatch(r"\[\s*(?:null|none|nan|n/a|na)?\s*\]",folded):
        return False
    return True
def clean_markup(value: object) -> str:
    return str(value or "").replace("\r\n", "\n").replace("\r", "\n").strip()
def row_references(row):
    return (), ()

context_count=int(test.context.map(has_context).sum())
if len(test)==2516 and context_count!=1361:
    raise RuntimeError(f"Phase1 lane canary failed: expected context=1361 closed=1155; got context={context_count} closed={len(test)-context_count}")
print("MORICHIKA test rows:", len(test), "context:", context_count, "closed:", len(test)-context_count)


In [ ]:
EXPECTED_DATASET_ID = "ishtyy/morichika-phase2-retrieval-strict-v3-20260720"
EXPECTED_MANIFEST_ID = "bfd6eecb3d011462d45282aa79967cd5258a90c6c5324193b5bb28d5a3c439b8"
EXPECTED_PACKAGE_ID = "6f68b6f284d55bacc3a34530a8492a78347b5c01243d55d256eb467ea096ad7e"

def file_sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def canonical_json(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))

def validate_manifest(manifest: dict) -> None:
    core = {k: v for k, v in manifest.items() if k != "manifest_id"}
    if manifest.get("dataset_id") != EXPECTED_DATASET_ID:
        raise RuntimeError("wrong MORICHIKA retrieval dataset")
    if manifest.get("manifest_id") != EXPECTED_MANIFEST_ID or hashlib.sha256(canonical_json(core).encode()).hexdigest() != EXPECTED_MANIFEST_ID:
        raise RuntimeError("MORICHIKA manifest identity mismatch")
    if manifest.get("package_id") != EXPECTED_PACKAGE_ID:
        raise RuntimeError("MORICHIKA package identity mismatch")
    files = manifest.get("files")
    if not isinstance(files, list) or len(files) < 90:
        raise RuntimeError("MORICHIKA payload declaration incomplete")

def verify_tree(root: Path, manifest: dict) -> Path:
    for spec in manifest["files"]:
        rel = PurePosixPath(str(spec["path"]))
        if rel.is_absolute() or ".." in rel.parts:
            raise RuntimeError(f"unsafe declared path: {rel}")
        path = root.joinpath(*rel.parts)
        if not path.is_file() or path.is_symlink():
            raise FileNotFoundError(f"declared MORICHIKA file missing: {rel}")
        if path.stat().st_size != int(spec["bytes"]) or file_sha256(path) != str(spec["sha256"]):
            raise RuntimeError(f"MORICHIKA hash/size mismatch: {rel}")
    return root

def materialize_morichika() -> tuple[Path, dict]:
    direct = []
    for path in INPUT_ROOT.rglob("bundle_manifest.json"):
        try:
            value = json.loads(path.read_text(encoding="utf-8-sig"))
        except Exception:
            continue
        if isinstance(value, dict) and value.get("manifest_id") == EXPECTED_MANIFEST_ID:
            direct.append((path, value))
    if len(direct) == 1:
        path, manifest = direct[0]
        validate_manifest(manifest)
        return verify_tree(path.parent, manifest), manifest
    if len(direct) > 1:
        raise RuntimeError(f"ambiguous MORICHIKA direct bundles: {len(direct)}")

    matches = []
    for archive_path in INPUT_ROOT.rglob("payload.zip"):
        try:
            with zipfile.ZipFile(archive_path) as archive:
                infos = archive.infolist()
                if len({i.filename for i in infos}) != len(infos):
                    raise RuntimeError("duplicate payload.zip members")
                for info in infos:
                    p = PurePosixPath(info.filename)
                    mode = (info.external_attr >> 16) & 0xFFFF
                    if p.is_absolute() or ".." in p.parts or "\\" in info.filename or info.flag_bits & 1 or stat.S_ISLNK(mode):
                        raise RuntimeError(f"unsafe payload.zip member: {info.filename}")
                    if p.name == "bundle_manifest.json" and info.file_size < 2 * 1024 * 1024:
                        value = json.loads(archive.read(info).decode("utf-8-sig"))
                        if value.get("manifest_id") == EXPECTED_MANIFEST_ID:
                            matches.append((archive_path, info.filename, value))
        except zipfile.BadZipFile:
            continue
    if len(matches) != 1:
        raise RuntimeError(f"expected one hash-bound MORICHIKA payload.zip, found {len(matches)}")
    archive_path, manifest_name, manifest = matches[0]
    validate_manifest(manifest)
    prefix = PurePosixPath(manifest_name).parent
    declared = {str(s["path"]): s for s in manifest["files"]}
    mapped = {}
    with zipfile.ZipFile(archive_path) as archive:
        for info in archive.infolist():
            if info.is_dir():
                continue
            p = PurePosixPath(info.filename)
            if tuple(p.parts[:len(prefix.parts)]) != tuple(prefix.parts):
                raise RuntimeError(f"mixed payload prefix: {info.filename}")
            rel = PurePosixPath(*p.parts[len(prefix.parts):]).as_posix()
            if rel == "bundle_manifest.json":
                continue
            if rel == "dataset-metadata.json":
                metadata_value=json.loads(archive.read(info).decode("utf-8-sig"))
                if metadata_value.get("id") != EXPECTED_DATASET_ID:
                    raise RuntimeError("nested dataset metadata identity mismatch")
                continue
            if rel not in declared or rel in mapped:
                raise RuntimeError(f"undeclared/duplicate payload member: {rel}")
            mapped[rel] = info
        if set(mapped) != set(declared):
            raise RuntimeError("payload.zip is incomplete")
        stage = WORK_DIR / "morichika_strict_v3"
        if stage.exists(): shutil.rmtree(stage)
        stage.mkdir(parents=True)
        (stage / "bundle_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
        for rel, info in mapped.items():
            spec = declared[rel]
            if info.file_size != int(spec["bytes"]):
                raise RuntimeError(f"zip member size mismatch: {rel}")
            dst = stage.joinpath(*PurePosixPath(rel).parts)
            dst.parent.mkdir(parents=True, exist_ok=True)
            h = hashlib.sha256(); written = 0
            with archive.open(info) as src, dst.open("wb") as out:
                while True:
                    chunk = src.read(8 * 1024 * 1024)
                    if not chunk: break
                    out.write(chunk); h.update(chunk); written += len(chunk)
            if written != int(spec["bytes"]) or h.hexdigest() != spec["sha256"]:
                raise RuntimeError(f"zip member hash mismatch: {rel}")
    return verify_tree(stage, manifest), manifest

MORICHIKA_ROOT, MORICHIKA_MANIFEST = materialize_morichika()
sys.path.insert(0, str(MORICHIKA_ROOT))
print("MORICHIKA corpus root:", MORICHIKA_ROOT)
print("MORICHIKA package:", EXPECTED_PACKAGE_ID, "files:", len(MORICHIKA_MANIFEST["files"]))


In [ ]:
"""Context-only policy packet for the MORICHIKA generalized hybrid.

The module is intentionally self-contained so its exact source can be embedded
in an offline Kaggle notebook.  It never retrieves, labels, or consults a
competition row identifier.  It converts the supplied context, question and
candidate response into an auditable instruction packet for the verifier.
"""

from __future__ import annotations

import hashlib
import json
import re
import unicodedata
from typing import Any, Callable


VERSION = "morichika-context-policy-v4-full-coverage-map-aggregate"

# These are reasoning checks, not predicted classes.  Keeping the recovered
# inventory explicit prevents a later prompt edit from silently dropping an
# edge-case family.
CANONICAL_POLICY_FAMILIES = (
    "question_grounding_answerability_and_premise_validity",
    "exact_entity_relation_property_and_answer_type",
    "direct_support_contradiction_and_partial_containment",
    "supplied_definition_formula_theory_and_rule_application",
    "date_event_role_and_as_of_time",
    "year_age_duration_calendar_and_timezone",
    "bounded_arithmetic_units_ratio_percentage_and_order",
    "kinship_and_relational_composition",
    "creator_founder_user_operator_office_title_and_jurisdiction",
    "birthplace_residence_nationality_and_event_participation",
    "legal_definition_section_effective_date_minimum_maximum_and_frequency",
    "numeric_whole_component_range_extremum_ordinal_and_granularity",
    "negation_quantifier_comparator_modality_and_clause_scope",
    "antonym_idiom_prefix_register_etymology_and_guruchandali",
    "samas_sandhi_affix_spelling_natva_and_satva",
    "unicode_joiner_conjunct_digit_punctuation_ocr_and_word_break",
    "ambiguity_conflict_invalid_premise_and_no_world_rescue",
)

ENGINEERED_EVALUATION_CELLS = (
    "context_only_lane", "full_context_boundary", "answerable_supported",
    "answerable_refuted", "genuinely_missing_nei", "same_passage_different_question",
    "entity_swap", "relation_swap", "answer_type_swap", "creator_vs_user_operator",
    "birthplace_vs_residence_event_place", "partial_answer_extra_claim",
    "theory_application", "formula_operand_application", "age_vs_year",
    "relative_timeline", "event_phase_order", "kinship_composition",
    "unit_or_number_swap", "negation_scope", "quantifier_comparator_scope",
    "grammar_operation_swap", "lexical_exact_vs_semantic_near",
    "unicode_equivalent", "unicode_word_break_non_equivalent", "cross_window_conflict",
)

OPERATION_AXIS = (
    "antonym_lookup", "birth_date_slot", "entity_text_relation",
    "event_date_or_status_slot", "idiom_meaning_lookup",
    "legal_definition_or_threshold", "legal_effective_date", "legal_maximum_fine",
    "legal_maximum_imprisonment", "legal_minimum_meeting_frequency",
    "location_slot", "numeric_or_ordinal_slot", "prefix_origin_classification",
    "samas_taxonomy", "sandhi_formation",
)

_SIGNALS: tuple[tuple[str, str], ...] = (
    ("year_age_duration", r"ব[য়য়]স|জন্ম|বিবাহ|বিয়ে|বিয়ে|বছর|সাল|year|age"),
    ("calendar_date", r"তারিখ|কবে|দিন|মাস|জানুয়ারি|ফেব্রুয়ারি|মার্চ|ডিসেম্বর"),
    ("timeline_event_order", r"আগে|পরে|তারপর|পরবর্তীতে|প্রথমে|শেষে|ঘটে|ঘটেছিল"),
    ("relative_time_offset", r"(?:বছর|দিন|মাস)\s*(?:আগে|পরে)|after|before|later"),
    ("kinship_relation_composition", r"বাবা|পিতা|মা|মাতা|দাদা|পিতামহ|নানা|grandfather"),
    ("bounded_arithmetic", r"কত|যোগ|বিয়োগ|বিয়োগ|গুণ|ভাগ|শতাংশ|হিসাব|calculate"),
    ("unit_dimension", r"কিলোমিটার|মিটার|গ্রাম|লিটার|সেকেন্ড|মিনিট|ঘণ্টা|টাকা|শতাংশ"),
    ("negation_scope", r"(?:^|\s)(?:না|নয়|নয়|নেই|নি)(?:\s|$)|হয়নি|হয়নি|করেনি"),
    ("quantifier_scope", r"সব|সকল|কোনো|কোনও|কেবল|শুধু|প্রত্যেক|একটিও|কিছু"),
    ("comparator_scope", r"সর্বাধিক|সর্বনিম্ন|বেশি|কম|পূর্ববর্তী|পরবর্তী|minimum|maximum"),
    ("modality_clause_scope", r"পারে|উচিত|অবশ্যই|সম্ভব|শর্তে|যদি|হলে"),
    ("definition_theory_rule_application", r"সংজ্ঞা|তত্ত্ব|সূত্র|নিয়ম|নিয়ম|অর্থ"),
    ("antonym_exact_pair", r"বিপরীত\s*শব্দ|antonym"),
    ("idiom_exact_meaning", r"বাগধারা|প্রবাদ"),
    ("prefix_suffix_origin", r"উপসর্গ|প্রত্যয়|প্রত্যয়"),
    ("samas_exact_operation", r"সমাস"),
    ("sandhi_exact_operands", r"সন্ধি"),
    ("register_etymology_guruchandali", r"গুরুচণ্ডালী|তৎসম|তদ্ভব|ফারসি|সংস্কৃত"),
    ("natva_satva_spelling", r"ণত্ব|ষত্ব|বানান"),
    ("creator_user_operator_role", r"স্রষ্টা|প্রতিষ্ঠাতা|নির্মাতা|ব্যবহারকারী|পরিচালক"),
    ("birthplace_residence_event_place", r"জন্মস্থান|বাসস্থান|রাজধানী|কোথায়|কোথায়|স্থান"),
    ("legal_definition_section", r"আইন|ধারা|বিধি|সংজ্ঞা"),
    ("minimum_maximum_frequency", r"সর্বনিম্ন|সর্বোচ্চ|কমপক্ষে|বেশিরভাগ|বার"),
    ("ordinal_list_order", r"প্রথম|দ্বিতীয়|দ্বিতীয়|তৃতীয়|তৃতীয়|ক্রম|তম"),
)

_BN_DIGITS = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
_JOINERS = {"\u200c", "\u200d", "\ufeff"}
_FIXED_LEXICAL_SHELLS = {
    "antonym_lookup": re.compile(r"(?:বিপরীত\s*শব্দ|বিপরীতার্থক\s*শব্দ).*(?:কী|কি|কোন)", re.IGNORECASE),
    "idiom_meaning_lookup": re.compile(r"(?:বাগধারা|প্রবাদ).*(?:অর্থ|মানে).*(?:কী|কি|কোন)", re.IGNORECASE),
    "prefix_origin_classification": re.compile(r"উপসর্গ.*(?:উৎস|শ্রেণি|জাতীয়|জাতীয়|কোন)", re.IGNORECASE),
}


def _sha(value: str) -> str:
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def _canonical(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))


def comparison_view(value: str) -> str:
    """Bounded comparison only; never rewrites literal evidence."""
    return " ".join(unicodedata.normalize("NFC", value).translate(_BN_DIGITS).casefold().split())


def unicode_receipt(value: str) -> dict[str, Any]:
    nfc = unicodedata.normalize("NFC", value)
    return {
        "literal_sha256": _sha(value),
        "utf8_bytes": len(value.encode("utf-8")),
        "codepoints": len(value),
        "nfc_identical": nfc == value,
        "nfc_sha256": _sha(nfc),
        "joiner_positions": [index for index, char in enumerate(value) if char in _JOINERS],
        "replacement_character_positions": [index for index, char in enumerate(value) if char == "\ufffd"],
        "comparison_view_sha256": _sha(comparison_view(value)),
    }


def contextual_exact_lexical_policy(
    question: str, response: str, records: list[dict[str, Any]] | None
) -> dict[str, Any]:
    """Expose only the frozen exact lexical-shell exception, nonterminally."""
    folded_question = comparison_view(question)
    folded_response = comparison_view(response)
    operation = next(
        (name for name, pattern in _FIXED_LEXICAL_SHELLS.items() if pattern.search(folded_question)),
        None,
    )
    base = {
        "lookup_mode": "not_applicable",
        "operation": operation,
        "key": None,
        "source_ids": [],
        "record_sha256": [],
        "exact_operation": operation is not None,
        "exact_key": False,
        "exact_sense_register": False,
        "conflict": False,
        "conflict_status": "none",
        "generic_retrieval_invoked": False,
        "terminal_label_authority": False,
        "model_nonterminal": True,
        "evidence": [],
    }
    if operation is None:
        base["lookup_mode"] = "forbidden_for_ordinary_context"
        return base

    matched: list[dict[str, Any]] = []
    conflicting: list[dict[str, Any]] = []
    for record in records or []:
        if record.get("operation") != operation:
            continue
        terms = [str(record.get("term_key") or "")] + [str(v) for v in record.get("display_terms", [])]
        exact_terms = []
        for term in terms:
            key = comparison_view(term)
            if key and re.search(rf"(?<![\u0980-\u09FFA-Za-z0-9]){re.escape(key)}(?![\u0980-\u09FFA-Za-z0-9])", folded_question):
                exact_terms.append(key)
        if not exact_terms:
            continue
        sense_values = [str(record.get(name) or "") for name in ("sense", "register", "etymological_class")]
        required_scope = [comparison_view(value) for value in sense_values if value]
        exact_scope = not required_scope or all(value in folded_question for value in required_scope)
        candidate = {**record, "_matched_key": sorted(exact_terms, key=lambda value: (-len(value), value))[0], "_exact_scope": exact_scope}
        if record.get("conflict_status") == "none" and exact_scope:
            matched.append(candidate)
        else:
            conflicting.append(candidate)

    if not matched and not conflicting:
        base["lookup_mode"] = "exact_shell_no_exact_key_nonterminal"
        return base
    all_candidates = matched + conflicting
    keys = {value["_matched_key"] for value in all_candidates}
    accepted_sets = {
        tuple(sorted(comparison_view(v) for v in value.get("accepted_answers", []) if str(v).strip()))
        for value in matched
    }
    conflict = bool(conflicting) or len(keys) != 1 or len(accepted_sets) > 1
    base.update({
        "lookup_mode": "exact_hash_bound_lexical_policy" if not conflict else "exact_lexical_conflict_nei_nonterminal",
        "key": sorted(keys)[0] if len(keys) == 1 else None,
        "source_ids": sorted({str(value.get("source_id") or "") for value in all_candidates if value.get("source_id")}),
        "record_sha256": sorted({_sha(_canonical({k: v for k, v in value.items() if not str(k).startswith("_")})) for value in all_candidates}),
        "exact_key": len(keys) == 1,
        "exact_sense_register": all(value.get("_exact_scope") is True for value in all_candidates),
        "conflict": conflict,
        "conflict_status": "conflict" if conflict else "none",
    })
    if not conflict:
        for value in matched[:3]:
            accepted = [comparison_view(v) for v in value.get("accepted_answers", [])]
            contrast = [comparison_view(v) for v in value.get("contrast_answers", [])]
            base["evidence"].append({
                "source_id": value.get("source_id"),
                "operation": operation,
                "term_key": value.get("term_key"),
                "accepted_answers": value.get("accepted_answers", []),
                "contrast_answers": value.get("contrast_answers", []),
                "response_relation": (
                    "support_candidate" if folded_response in accepted else
                    "counter_candidate" if folded_response in contrast else "neutral_candidate"
                ),
            })
    return base


def detected_policy_families(context: str, question: str, response: str) -> list[str]:
    joined = "\n".join((question, response, context))
    detected = {
        "lane_context_only", "full_context_evidence_boundary", "question_answerability",
        "question_premise_validity", "exact_answer_slot", "same_passage_different_question",
        "entity_identity", "relation_property_identity", "answer_type_identity",
        "direct_support", "direct_contradiction", "partial_containment_extra_claim",
        "ambiguity_conflict_nei", "unicode_nfc_joiner_conjunct_digit", "ocr_word_break_caution",
    }
    for family, pattern in _SIGNALS:
        if re.search(pattern, joined, re.IGNORECASE):
            detected.add(family)
    if re.search(r"[০-৯0-9]", joined):
        detected.update({"number_whole_component_range", "formula_operand_application"})
    order = [name for name, _ in _SIGNALS]
    order += [
        "lane_context_only", "full_context_evidence_boundary", "question_answerability",
        "question_premise_validity", "exact_answer_slot", "same_passage_different_question",
        "entity_identity", "relation_property_identity", "answer_type_identity",
        "direct_support", "direct_contradiction", "partial_containment_extra_claim",
        "ambiguity_conflict_nei", "unicode_nfc_joiner_conjunct_digit", "ocr_word_break_caution",
        "number_whole_component_range", "formula_operand_application",
    ]
    return [family for family in dict.fromkeys(order) if family in detected]


def full_coverage_windows(
    context: str, *, max_chars: int = 3000, overlap_chars: int = 320
) -> list[dict[str, Any]]:
    """Partition every context character into overlapping replayable windows."""
    if not context:
        raise ValueError("context must be non-empty")
    if max_chars < 256 or overlap_chars < 32 or overlap_chars >= max_chars:
        raise ValueError("invalid full-coverage window geometry")
    windows: list[dict[str, Any]] = []
    start = 0
    while start < len(context):
        hard_end = min(len(context), start + max_chars)
        end = hard_end
        if hard_end < len(context):
            boundaries = [context.rfind(mark, start + max_chars // 2, hard_end) for mark in ("\n", "।", ".", "?", "!")]
            boundary = max(boundaries)
            if boundary >= start:
                end = boundary + 1
        literal = context[start:end]
        windows.append({
            "window_index": len(windows),
            "context_char_start": start,
            "context_char_end": end,
            "literal_text": literal,
            "literal_span_sha256": _sha(literal),
        })
        if end == len(context):
            break
        start = end - overlap_chars
    covered = [False] * len(context)
    for index, window in enumerate(windows):
        start, end = window["context_char_start"], window["context_char_end"]
        literal = window["literal_text"]
        if context[start:end] != literal or _sha(literal) != window["literal_span_sha256"]:
            raise ValueError(f"full-coverage window {index} does not replay")
        for position in range(start, end):
            covered[position] = True
    if not all(covered):
        raise ValueError("full-coverage window ledger contains a character gap")
    return windows


def build_window_adjudication_user(
    window: dict[str, Any], question: str, response: str, total_windows: int
) -> str:
    return (
        "WINDOW-LOCAL CONTEXT PASS. Judge only evidence present in this literal supplied-context window. "
        "A local miss is not a contradiction: output not_enough_information unless this window directly "
        "supports or directly refutes the exact question slot. Do not use outside knowledge.\n\n"
        f"QUESTION (literal):\n{question}\n\nCANDIDATE RESPONSE (literal):\n{response}\n\n"
        f"WINDOW {window['window_index'] + 1}/{total_windows} "
        f"[chars={window['context_char_start']}:{window['context_char_end']}; "
        f"sha256={window['literal_span_sha256']}]:\n{window['literal_text']}"
    )


def _literal_question_excerpt(
    window: dict[str, Any], question: str, max_chars: int
) -> dict[str, Any]:
    """Select a literal, question-conditioned excerpt without rewriting it."""
    literal = str(window["literal_text"])
    if max_chars < 32:
        max_chars = 32
    q_tokens = {
        token.casefold() for token in re.findall(r"[\u0980-\u09FFA-Za-z0-9]+", question)
        if len(token) > 1
    }
    candidates = list(re.finditer(r"[^।.!?\n]+[।.!?]?", literal))
    if candidates:
        def score(match: re.Match[str]) -> tuple[int, int]:
            tokens = {
                token.casefold() for token in re.findall(r"[\u0980-\u09FFA-Za-z0-9]+", match.group())
            }
            return (len(tokens & q_tokens), -match.start())
        best = max(candidates, key=score)
        local_start = max(0, best.start() - max_chars // 4)
        local_end = min(len(literal), local_start + max_chars)
    else:
        local_start, local_end = 0, min(len(literal), max_chars)
    excerpt = literal[local_start:local_end]
    absolute_start = int(window["context_char_start"]) + local_start
    absolute_end = absolute_start + len(excerpt)
    return {
        "excerpt_char_start": absolute_start,
        "excerpt_char_end": absolute_end,
        "literal_excerpt": excerpt,
        "literal_excerpt_sha256": _sha(excerpt),
    }


def build_aggregation_user(
    question: str,
    response: str,
    window_results: list[dict[str, Any]],
    *,
    selected_notes: list[dict[str, Any]] | None = None,
    bounded_derivations: list[dict[str, Any]] | None = None,
    lexical_policy: dict[str, Any] | None = None,
) -> str:
    """Create the exact-question final adjudication over all window passes."""
    compact = [
        {
            "window_index": row["window_index"],
            "context_char_start": row["context_char_start"],
            "context_char_end": row["context_char_end"],
            "literal_span_sha256": row["literal_span_sha256"],
            "window_verdict": row["window_verdict"],
            "literal_excerpt": row.get("literal_excerpt", ""),
            "excerpt_char_start": row.get("excerpt_char_start"),
            "excerpt_char_end": row.get("excerpt_char_end"),
            "literal_excerpt_sha256": row.get("literal_excerpt_sha256"),
        }
        for row in window_results
    ]
    notes = list(selected_notes or [])[:32]
    derivations = list(bounded_derivations or [])[:8]
    return (
        "FINAL FULL-CONTEXT AGGREGATION. The window ledger collectively covered every supplied-context "
        "character. Rebind the exact question and candidate response. A not_enough_information window means "
        "only that its local span was silent; it is never counter-evidence. If any aligned window refutes the "
        "response, do not let an unrelated support override it. Unresolved support/refutation across different "
        "entities, slots, event phases, dates, operations, or ambiguous coreference is not_enough_information. "
        "Use no outside knowledge.\n\n"
        f"QUESTION (literal):\n{question}\n\nCANDIDATE RESPONSE (literal):\n{response}\n\n"
        "PER-WINDOW STRUCTURED RESULTS AND LITERAL EXCERPTS:\n" + _canonical(compact)
        + "\n\nFULL-CONTEXT ROUTER SOURCE-LINKED NOTES (advisory; offsets/hashes bind them to supplied context):\n"
        + _canonical(notes)
        + "\n\nBOUNDED DERIVATION CANDIDATES (advisory; verify operator, operands, slot and unit):\n"
        + _canonical(derivations)
        + "\n\nFROZEN EXACT LEXICAL-SHELL POLICY (nonterminal; ordinary retrieval forbidden):\n"
        + _canonical(lexical_policy or {"lookup_mode": "not_applicable", "generic_retrieval_invoked": False})
    )


def build_contextual_policy_packet(
    context: str,
    question: str,
    response: str,
    router: Callable[..., dict[str, Any]],
    lexical_records: list[dict[str, Any]] | None = None,
    *,
    max_windows: int = 8,
    full_context_char_limit: int = 6000,
    coverage_window_chars: int = 3000,
    coverage_overlap_chars: int = 320,
) -> tuple[str, dict[str, Any]]:
    """Build a context-only prompt plus restart-safe diagnostics."""
    if not all(isinstance(value, str) for value in (context, question, response)):
        raise TypeError("context, question and response must be strings")
    if not context.strip() or not question.strip():
        raise ValueError("context and question must be non-empty")

    route = router(context, question, response, max_windows=max_windows)
    if route.get("external_retrieval_allowed") is not False:
        raise ValueError("context router must explicitly forbid external retrieval")
    if route.get("context_sha256") != _sha(context):
        raise ValueError("context router hash mismatch")

    window_calls: list[dict[str, Any]] = []
    if len(context) <= full_context_char_limit:
        evidence = context
        evidence_mode = "complete_literal_supplied_context"
        full_context_inference_coverage = True
    else:
        coverage = full_coverage_windows(
            context,
            max_chars=coverage_window_chars,
            overlap_chars=coverage_overlap_chars,
        )
        excerpt_chars = max(48, min(320, 3200 // len(coverage)))
        window_calls = [
            {
                **{key: window[key] for key in (
                    "window_index", "context_char_start", "context_char_end", "literal_span_sha256"
                )},
                **_literal_question_excerpt(window, question, excerpt_chars),
                "user": build_window_adjudication_user(window, question, response, len(coverage)),
            }
            for window in coverage
        ]
        # The ordinary single-call payload is never used for a long context;
        # it is deliberately tiny so no truncated projection can be mistaken
        # for the evidence universe.
        evidence = "[FULL-COVERAGE WINDOW PASSES REQUIRED BEFORE AGGREGATION]"
        evidence_mode = "all_literal_windows_then_final_aggregation"
        full_context_inference_coverage = True

    signals = detected_policy_families(context, question, response)
    lexical_policy = contextual_exact_lexical_policy(question, response, lexical_records)
    aggregation_derivations = [
        value for value in list(route.get("bounded_derivation_candidates") or [])[:8]
        if isinstance(value, dict) and value.get("source_linked") is True
    ]
    operand_ids = {
        str(note_id)
        for value in aggregation_derivations
        for note_id in value.get("operand_note_ids", [])
    }
    routed_notes = list(route.get("selected_notes") or [])
    routed_notes.sort(key=lambda note: (str(note.get("note_id")) not in operand_ids, int(note.get("context_char_start", 0))))
    aggregation_notes = []
    note_budget = 2400
    used_note_chars = 0
    for note in routed_notes[:32]:
        start = int(note["context_char_start"])
        end = int(note["context_char_end"])
        literal = str(note["literal_text"])
        if context[start:end] != literal or _sha(literal) != note["literal_span_sha256"]:
            raise ValueError("router selected note does not replay against supplied context")
        serialized_chars = len(_canonical(note))
        if aggregation_notes and used_note_chars + serialized_chars > note_budget:
            continue
        aggregation_notes.append(note)
        used_note_chars += serialized_chars
    notes = {
        "selected_notes": aggregation_notes,
        "bounded_derivation_candidates": aggregation_derivations,
    }
    contract = {
        "version": VERSION,
        "evidence_universe": "only_the_literal_supplied_context",
        "external_retrieval_allowed": False,
        "outside_world_fact_rescue_allowed": False,
        "evidence_mode": evidence_mode,
        "full_context_inference_coverage": full_context_inference_coverage,
        "every_context_character_processed": True,
        "detected_policy_families": signals,
        "checks_in_order": [
            "validate question premise and whether the requested slot is answerable",
            "bind exact entity relation property answer type event phase time polarity comparator and unit",
            "seek direct support and direct contradiction for that exact slot",
            "apply only supplied definition theory rule formula and bounded operands",
            "separate refuted from missing ambiguous conflicting or locally silent window evidence",
            "compare Unicode only through bounded NFC digit joiner and attested word-break views; near-looking forms are not automatically equal",
        ],
        "verdict_rule": {
            "supported": "the complete response follows for the exact requested slot",
            "refuted": "the supplied context establishes an incompatible value or claim",
            "not_enough_information": "required evidence is absent omitted ambiguous conflicting or premise-invalid",
        },
        "fixed_lexical_exception": lexical_policy,
    }
    user = (
        "CONTEXT-ONLY VERIFICATION CONTRACT:\n" + _canonical(contract)
        + "\n\nQUESTION (literal):\n" + question
        + "\n\nSUPPLIED CONTEXT EVIDENCE (literal; this is the only evidence universe):\n" + evidence
        + "\n\nSOURCE-LINKED EXTRACTIVE NOTES (advisory, never new evidence):\n" + _canonical(notes)
        + "\n\nCANDIDATE RESPONSE (verify every material claim):\n" + response
    )
    diagnostic = {
        **route,
        "context_policy_version": VERSION,
        "policy_family_inventory_count": len(CANONICAL_POLICY_FAMILIES),
        "canonical_policy_family_count": len(CANONICAL_POLICY_FAMILIES),
        "engineered_evaluation_cell_count": len(ENGINEERED_EVALUATION_CELLS),
        "operation_axis_count": len(OPERATION_AXIS),
        "canonical_policy_families": list(CANONICAL_POLICY_FAMILIES),
        "engineered_evaluation_cells": list(ENGINEERED_EVALUATION_CELLS),
        "operation_axis": list(OPERATION_AXIS),
        "detected_policy_families": signals,
        "evidence_mode": evidence_mode,
        "full_context_inference_coverage": full_context_inference_coverage,
        "context_literal": unicode_receipt(context),
        "question_literal": unicode_receipt(question),
        "response_literal": unicode_receipt(response),
        "prompt_sha256": _sha(user),
        "requires_window_aggregation": bool(window_calls),
        "window_calls": window_calls,
        "window_count": len(window_calls) if window_calls else 1,
        "full_context_character_count": len(context),
        "aggregation_selected_notes": aggregation_notes,
        "aggregation_bounded_derivations": aggregation_derivations,
        "contextual_lexical_policy": lexical_policy,
    }
    diagnostic.pop("receipt_sha256", None)
    diagnostic["receipt_sha256"] = _sha(_canonical(diagnostic))
    return user, diagnostic


__all__ = [
    "VERSION", "CANONICAL_POLICY_FAMILIES", "ENGINEERED_EVALUATION_CELLS",
    "OPERATION_AXIS", "comparison_view", "unicode_receipt",
    "detected_policy_families", "full_coverage_windows",
    "build_window_adjudication_user", "build_aggregation_user",
    "build_contextual_policy_packet",
]


In [ ]:
"""Strict typed grammar lookup repair for MORICHIKA contextual verification.

The Phase 2 contextual lane remains context-only by default.  A bounded
exception is made only when the *question itself* requests a recognized
Bengali grammar, lexical, spelling, or language-theory operation.  Such a row
is represented by a synthetic closed-book proxy for retrieval, then filtered
back to exact-normalized, operation/key/slot-aligned evidence.  The evidence
is advisory and can never produce a terminal label.

This file can be imported normally, or executed immediately after
``contextual_policy_v4_runtime.py`` in one self-contained Kaggle notebook.
"""

from __future__ import annotations

import hashlib
import json
import re
from pathlib import Path
from typing import Any, Callable


# The generated notebook executes the complete v4 runtime first.  Normal
# package imports take the except branch instead.  This avoids relying on an
# undeclared Kaggle input module while retaining one tested implementation of
# full-context windowing and aggregation.
try:  # pragma: no branch - the branch is environment-dependent by design
    _V4_VERSION = VERSION  # type: ignore[name-defined]
    _V4_CANONICAL_POLICY_FAMILIES = CANONICAL_POLICY_FAMILIES  # type: ignore[name-defined]
    _V4_ENGINEERED_EVALUATION_CELLS = ENGINEERED_EVALUATION_CELLS  # type: ignore[name-defined]
    _V4_OPERATION_AXIS = OPERATION_AXIS  # type: ignore[name-defined]
    _v4_build_aggregation_user = build_aggregation_user  # type: ignore[name-defined]
    _v4_build_contextual_policy_packet = build_contextual_policy_packet  # type: ignore[name-defined]
    comparison_view = comparison_view  # type: ignore[name-defined,self-assigning-variable]
    unicode_receipt = unicode_receipt  # type: ignore[name-defined,self-assigning-variable]
    detected_policy_families = detected_policy_families  # type: ignore[name-defined,self-assigning-variable]
    full_coverage_windows = full_coverage_windows  # type: ignore[name-defined,self-assigning-variable]
    build_window_adjudication_user = build_window_adjudication_user  # type: ignore[name-defined,self-assigning-variable]
except NameError:
    from pipeline.contextual_policy_v4_runtime import (
        CANONICAL_POLICY_FAMILIES as _V4_CANONICAL_POLICY_FAMILIES,
        ENGINEERED_EVALUATION_CELLS as _V4_ENGINEERED_EVALUATION_CELLS,
        OPERATION_AXIS as _V4_OPERATION_AXIS,
        VERSION as _V4_VERSION,
        build_aggregation_user as _v4_build_aggregation_user,
        build_contextual_policy_packet as _v4_build_contextual_policy_packet,
        build_window_adjudication_user,
        comparison_view,
        detected_policy_families,
        full_coverage_windows,
        unicode_receipt,
    )


VERSION = "morichika-context-policy-v5-strict-typed-grammar-lookup"
CANONICAL_POLICY_FAMILIES = tuple(_V4_CANONICAL_POLICY_FAMILIES)
ENGINEERED_EVALUATION_CELLS = tuple(_V4_ENGINEERED_EVALUATION_CELLS) + (
    "typed_grammar_exact_reference",
    "typed_grammar_wrong_operation_quarantine",
    "typed_grammar_retrieval_miss_nonrefutation",
)
TYPED_LOOKUP_OPERATIONS = (
    "antonym_lookup",
    "idiom_meaning_lookup",
    "prefix_origin_classification",
    "suffix_origin_classification",
    "sandhi_formation",
    "samas_taxonomy",
    "one_word_expression",
    "pair_register_guruchandali",
    "natva_satva_rule",
    "spelling_rule",
    "grammar_theory_rule",
)
OPERATION_AXIS = tuple(dict.fromkeys((*_V4_OPERATION_AXIS, *TYPED_LOOKUP_OPERATIONS)))

PROXY_PREFIX = "__morichika_typed_context_lookup_v5__"
SOURCE_PRIORITY_CLASSES = (
    "books_and_user_ocr_priority",
    "curated_dataset",
    "other_authorized_evidence",
    "community_encyclopedia_corroboration",
)
_SOURCE_PRIORITY_BY_ID = {
    "downloads_bcs_10_50": (0, SOURCE_PRIORITY_CLASSES[0]),
    "joykoli_six_part": (0, SOURCE_PRIORITY_CLASSES[0]),
    "nctb_passage_qa_mendeley_v1": (0, SOURCE_PRIORITY_CLASSES[0]),
    "nctb_schooltext": (0, SOURCE_PRIORITY_CLASSES[0]),
    "bangla_mmlu": (1, SOURCE_PRIORITY_CLASSES[1]),
    "nctb_education_aux": (1, SOURCE_PRIORITY_CLASSES[1]),
    "nctb_qa_87805": (1, SOURCE_PRIORITY_CLASSES[1]),
    "nctbench": (1, SOURCE_PRIORITY_CLASSES[1]),
    "openai_mmmlu_bn": (1, SOURCE_PRIORITY_CLASSES[1]),
    "squad_bn": (1, SOURCE_PRIORITY_CLASSES[1]),
    "uddipok_v3": (1, SOURCE_PRIORITY_CLASSES[1]),
    "lexical::bangla_bagdhara_cc_by_sa_4": (1, SOURCE_PRIORITY_CLASSES[1]),
    "bangla_bagdhara_cc_by_sa_4": (1, SOURCE_PRIORITY_CLASSES[1]),
    "lexical::bnwiktionary_cc_by_sa_4_20260701": (2, SOURCE_PRIORITY_CLASSES[2]),
    "bnwiktionary_cc_by_sa_4_20260701": (2, SOURCE_PRIORITY_CLASSES[2]),
    "bangla_wordnet": (2, SOURCE_PRIORITY_CLASSES[2]),
    "bengali_wikipedia": (3, SOURCE_PRIORITY_CLASSES[3]),
    "bengali_wikipedia_20210320": (3, SOURCE_PRIORITY_CLASSES[3]),
}


# Specific operations precede the general grammar/theory shell.  Detection is
# question-only: a grammar word elsewhere in a reused passage must never route
# a different question to external lookup.
_OPERATION_PATTERNS: tuple[tuple[str, re.Pattern[str]], ...] = (
    ("antonym_lookup", re.compile(
        r"বিপরীত(?:ার্থক)?\s*শব্দ|শুদ্ধ\s*বিপরীত|\S+\s*[-–—]?\s*এর\s*বিপরীত", re.I)),
    ("idiom_meaning_lookup", re.compile(
        r"বাগ\s*ধারা|বাগধারা|প্রবাদ(?:টির|টির)?\s*(?:অর্থ|মানে)|প্রবাদ\s*[-/]\s*প্রবচন", re.I)),
    ("prefix_origin_classification", re.compile(
        r"উপসর্গ(?:টি|টির)?\s*(?:কোন|কী|কি).{0,30}(?:শ্রেণি|উৎস|জাতীয়|জাতীয়|ভাষা)|"
        r"শব্দ(?:ে|টির|টিতে|ের).{0,32}উপসর্গ|উপসর্গ\s*যোগে", re.I)),
    ("suffix_origin_classification", re.compile(
        r"প্রত্য(?:য়|য়)(?:টি|টির)?\s*(?:কোন|কী|কি).{0,30}(?:শ্রেণি|উৎস|জাতীয়|জাতীয়|ভাষা)|"
        r"(?:শব্দ|প্রকৃতি|ধাতু)(?:তে|টির|ের)?.{0,32}প্রত্য(?:য়|য়)|কৃৎ\s*প্রত্য(?:য়|য়)|"
        r"তদ্ধিত\s*প্রত্য(?:য়|য়)", re.I)),
    ("sandhi_formation", re.compile(
        r"(?<![\u0980-\u09FF])সন্ধি(?:টি|টির|র|তে)?\s*(?:বিচ্ছেদ|কী|কি|কোন|কর|হয়|হয়|জাত|নির্ণ)|"
        r"(?<![\u0980-\u09FF])সন্ধিবিচ্ছেদ|(?:কোন|কী|কি)\s*(?:ধরনের\s*)?সন্ধি", re.I)),
    ("samas_taxonomy", re.compile(
        r"(?<![\u0980-\u09FF])সমাস(?:টি|টির|ের|ে)?\s*(?:কী|কি|কাকে|কোন|শব্দ|বদ্ধ|নির্ণ|উদাহরণ|অর্থ)|"
        r"(?:কোন|কী|কি)\s*(?:ধরনের\s*|প্রকারের\s*)?সমাস|ব্যাস\s*বাক্য", re.I)),
    ("one_word_expression", re.compile(
        r"এক\s*কথা(?:য়|য়)\s*প্রকাশ|এককথা(?:য়|য়)\s*প্রকাশ|বাক্য\s*সংকোচন", re.I)),
    ("pair_register_guruchandali", re.compile(
        r"গুরুচণ্ডালী|গুরু\s*চণ্ডালী|সাধু\s*(?:ও|ও\s*)?চলিত|তৎসম\s*(?:ও|-)\s*তদ্ভব|"
        r"শব্দ\s*(?:জোড়া|জোড়া|যুগল)|যুগ্ম\s*শব্দ|পারিভাষিক\s*শব্দ", re.I)),
    ("natva_satva_rule", re.compile(r"ণ\s*[-–—]?\s*ত্ব|ষ\s*[-–—]?\s*ত্ব|ণত্ব|ষত্ব", re.I)),
    ("spelling_rule", re.compile(r"শুদ্ধ\s*বানান|বানান\s*(?:বিধি|রীতি|নিয়ম|নিয়ম)", re.I)),
    ("grammar_theory_rule", re.compile(
        r"(?:ব্যাকরণ|ভাষাতত্ত্ব|শব্দতত্ত্ব).*(?:সংজ্ঞা|তত্ত্ব|নিয়ম|নিয়ম|বিধান|গঠন|শ্রেণি|প্রকার)|"
        r"(?:সংজ্ঞা|তত্ত্ব|নিয়ম|নিয়ম|বিধান).*(?:ব্যাকরণ|ভাষাতত্ত্ব|শব্দতত্ত্ব)", re.I)),
)

_TERM_BOUNDARY = r"[\u0980-\u09FFA-Za-z0-9]"


def _sha(value: str) -> str:
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def _canonical(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))


def detect_contextual_lookup_request(question: str) -> dict[str, Any]:
    """Classify a narrow typed lookup from the literal question only."""

    if not isinstance(question, str):
        raise TypeError("question must be a string")
    folded = comparison_view(question)
    matches = [name for name, pattern in _OPERATION_PATTERNS if pattern.search(folded)]
    operation = matches[0] if matches else None
    return {
        "applicable": operation is not None,
        "operation": operation,
        "all_detected_operations": matches,
        "question_sha256": _sha(question),
        "detection_scope": "literal_question_only",
        "ordinary_context_external_lookup_allowed": False,
        "terminal_label_authority": False,
    }


def _proxy_id(example_id: str, question: str, operation: str) -> str:
    payload = _canonical({"example_id": example_id, "question": question, "operation": operation})
    return PROXY_PREFIX + _sha(payload)[:32]


def prepare_contextual_typed_lookup_input(
    rows: list[dict[str, Any]], output_path: Path
) -> tuple[Path, dict[str, dict[str, str]]]:
    """Append label-free closed-book proxies for typed contextual questions.

    Original contextual rows remain present and retain ``context_available``;
    the shared retriever therefore continues to reject them.  Only synthetic
    proxy rows enter the strict-v3 closed-book retrieval lane.
    """

    output_path = Path(output_path)
    original_ids = {str(row.get("example_id", "")) for row in rows}
    if "" in original_ids or len(original_ids) != len(rows):
        raise ValueError("input rows require unique non-empty example_id values")
    combined = [dict(row) for row in rows]
    proxy_map: dict[str, dict[str, str]] = {}
    for row in rows:
        if row.get("context_available") is not True:
            continue
        question = str(row.get("model_prompt_bn") or "")
        request = detect_contextual_lookup_request(question)
        if request["applicable"] is not True:
            continue
        operation = str(request["operation"])
        proxy_id = _proxy_id(str(row["example_id"]), question, operation)
        if proxy_id in original_ids or proxy_id in proxy_map:
            raise ValueError("typed contextual proxy ID collision")
        proxy_map[proxy_id] = {
            "example_id": str(row["example_id"]),
            "operation": operation,
            "question_sha256": str(request["question_sha256"]),
        }
        combined.append({
            "example_id": proxy_id,
            "source_index": int(row.get("source_index", 0)),
            "model_prompt_bn": question,
            "model_response_bn": str(row.get("model_response_bn") or ""),
            "model_context_bn": "",
            "context_available": False,
            "context_policy_route": {
                "operation": operation,
                "typed_context_lookup_proxy": True,
                "terminal_label_authority": False,
            },
            "formatting_provenance": {
                "status": "typed_context_lookup_proxy_no_context_no_label",
            },
        })
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = output_path.with_suffix(output_path.suffix + ".tmp")
    temporary.write_text(
        "".join(_canonical(row) + "\n" for row in combined), encoding="utf-8"
    )
    temporary.replace(output_path)
    return output_path, proxy_map


def attach_contextual_typed_lookup(
    retrieved: dict[str, dict[str, Any]],
    proxy_map: dict[str, dict[str, str]],
    rows: list[dict[str, Any]] | None = None,
) -> dict[str, dict[str, Any]]:
    """Attach proxy evidence to original IDs and remove every proxy ID.

    Upstream terminal candidates are deliberately discarded for this lane.
    The v5 policy filter, not the closed-book terminal adapter, controls which
    exact candidates may be shown to the contextual verifier.
    """

    result = {
        str(key): dict(value)
        for key, value in retrieved.items()
        if str(key) not in proxy_map
    }
    for proxy_id, binding in proxy_map.items():
        if proxy_id not in retrieved:
            raise ValueError(f"typed lookup proxy missing from retrieval output: {proxy_id}")
        original_id = binding["example_id"]
        if original_id not in result:
            raise ValueError(f"typed lookup original missing from retrieval output: {original_id}")
        proxy_item = retrieved[proxy_id]
        result[original_id]["contextual_typed_lookup"] = {
            "version": VERSION,
            "proxy_id": proxy_id,
            "operation": binding["operation"],
            "question_sha256": binding["question_sha256"],
            "retrieval_candidates": list(proxy_item.get("retrieval_candidates") or []),
            "retrieval_audit_quarantined_candidate_count": len(
                proxy_item.get("retrieval_audit_quarantined_candidates") or []
            ),
            "upstream_merged_status": str(
                (proxy_item.get("merged_source_candidate") or {}).get("status") or ""
            ),
            "terminal_label_authority": False,
        }
    if rows is not None:
        expected = {str(row["example_id"]) for row in rows}
        if set(result) != expected:
            raise ValueError("retrieval mapping does not exactly cover original rows")
        typed_originals = {value["example_id"] for value in proxy_map.values()}
        for row in rows:
            example_id = str(row["example_id"])
            if row.get("context_available") is True and example_id not in typed_originals:
                if result[example_id].get("contextual_typed_lookup") is not None:
                    raise ValueError("ordinary contextual row received external lookup")
    return result


def _exact_term_in_question(term: object, folded_question: str) -> str | None:
    key = comparison_view(str(term or ""))
    if not key:
        return None
    pattern = rf"(?<!{_TERM_BOUNDARY}){re.escape(key)}(?!{_TERM_BOUNDARY})"
    return key if re.search(pattern, folded_question) else None


def _response_relation(response: str, answers: list[Any], choices: list[Any]) -> str:
    response_key = comparison_view(response)
    answer_keys = {comparison_view(str(value)) for value in answers if str(value).strip()}
    choice_keys = {comparison_view(str(value)) for value in choices if str(value).strip()}
    if response_key and response_key in answer_keys:
        return "support_candidate"
    if response_key and response_key in choice_keys - answer_keys:
        return "counter_candidate"
    return "neutral_candidate"


def _lexical_exact_evidence(
    operation: str, question: str, response: str,
    records: list[dict[str, Any]] | None,
) -> list[dict[str, Any]]:
    folded_question = comparison_view(question)
    evidence: list[dict[str, Any]] = []
    for record in records or []:
        if str(record.get("operation") or "") != operation:
            continue
        terms = [record.get("term_key")] + list(record.get("display_terms") or [])
        keys = [key for term in terms if (key := _exact_term_in_question(term, folded_question))]
        if not keys or str(record.get("conflict_status") or "none") != "none":
            continue
        scopes = [
            comparison_view(str(record.get(name) or ""))
            for name in ("sense", "register", "etymological_class")
            if str(record.get(name) or "").strip()
        ]
        if scopes and not all(scope in folded_question for scope in scopes):
            continue
        accepted = list(record.get("accepted_answers") or [])
        contrasts = list(record.get("contrast_answers") or [])
        source_id = str(record.get("source_id") or "")
        source_priority, authority_class = _SOURCE_PRIORITY_BY_ID.get(
            source_id, (2, SOURCE_PRIORITY_CLASSES[2])
        )
        core = {
            "source_id": source_id,
            "source_record_index": int(record.get("source_record_index", -1)),
            "source_priority": source_priority,
            "authority_class": authority_class,
            "operation": operation,
            "exact_normalized": True,
            "key": sorted(set(keys), key=lambda value: (-len(value), value))[0],
            "answers": accepted,
            "choices": contrasts,
            "response_relation": _response_relation(response, accepted, contrasts),
            "alignment": "exact_operation_term_scope_and_answer_slot",
            "terminal_label_authority": False,
            "wikipedia_corroboration_only": False,
        }
        core["evidence_sha256"] = _sha(_canonical(core))
        evidence.append(core)
    return evidence


def _candidate_priority(candidate: dict[str, Any]) -> tuple[int, str]:
    source_id = str(candidate.get("source_id") or "")
    fallback_tier, fallback_class = _SOURCE_PRIORITY_BY_ID.get(
        source_id, (2, SOURCE_PRIORITY_CLASSES[2])
    )
    tier = int(candidate.get("authority_tier", fallback_tier))
    authority_class = str(candidate.get("authority_class") or fallback_class)
    if authority_class not in SOURCE_PRIORITY_CLASSES:
        tier, authority_class = fallback_tier, fallback_class
    return tier, authority_class


def _strict_retrieval_evidence(
    request: dict[str, Any], question: str, response: str,
    typed_lookup_item: dict[str, Any] | None,
) -> tuple[list[dict[str, Any]], dict[str, int]]:
    counters = {
        "examined": 0, "accepted": 0, "fuzzy_rejected": 0,
        "operation_rejected": 0, "alignment_rejected": 0,
    }
    if not typed_lookup_item:
        return [], counters
    binding = typed_lookup_item.get("contextual_typed_lookup") or typed_lookup_item
    operation = str(request["operation"])
    if (
        str(binding.get("operation") or "") != operation
        or str(binding.get("question_sha256") or "") != _sha(question)
    ):
        counters["alignment_rejected"] += len(binding.get("retrieval_candidates") or [])
        return [], counters
    accepted: list[dict[str, Any]] = []
    for candidate in binding.get("retrieval_candidates") or []:
        counters["examined"] += 1
        if candidate.get("exact_normalized") is not True:
            counters["fuzzy_rejected"] += 1
            continue
        source_text = str(candidate.get("question") or "").strip()
        if not source_text:
            source_text = str(candidate.get("supporting_text") or "")
        source_request = detect_contextual_lookup_request(source_text)
        if source_request.get("operation") != operation:
            counters["operation_rejected"] += 1
            continue
        if not (
            candidate.get("model_facing_eligible") is True
            and int(candidate.get("semantic_alignment_tier", 99)) == 0
            and int(candidate.get("slot_compatibility_tier", 99)) == 0
            and int(candidate.get("policy_compatibility_tier", 99)) in (0, 1)
            and candidate.get("number_set_match") is not False
            and candidate.get("negation_set_match") is not False
        ):
            counters["alignment_rejected"] += 1
            continue
        declared_query_operation = str(candidate.get("query_policy_operation") or "")
        if declared_query_operation and declared_query_operation != operation:
            counters["operation_rejected"] += 1
            continue
        tier, authority_class = _candidate_priority(candidate)
        answers = list(candidate.get("answers") or [])
        choices = list(candidate.get("choices") or [])
        compact = {
            "source_id": str(candidate.get("source_id") or ""),
            "source_record_index": int(candidate.get("source_record_index", -1)),
            "source_priority": tier,
            "authority_class": authority_class,
            "operation": operation,
            "exact_normalized": True,
            "key_alignment": "exact_normalized_full_question",
            "entity_alignment": "exact_normalized_full_question",
            "slot_alignment": "upstream_tier_zero",
            "question": str(candidate.get("question") or ""),
            "supporting_text": str(candidate.get("supporting_text") or "")[:640],
            "answers": answers[:8],
            "choices": choices[:12],
            "response_relation": _response_relation(response, answers, choices),
            "terminal_label_authority": False,
            "wikipedia_corroboration_only": authority_class
            == "community_encyclopedia_corroboration",
        }
        compact["evidence_sha256"] = _sha(_canonical(compact))
        accepted.append(compact)
        counters["accepted"] += 1
    accepted.sort(key=lambda value: (
        int(value["source_priority"]),
        0 if value["response_relation"] in ("support_candidate", "counter_candidate") else 1,
        str(value["source_id"]), int(value["source_record_index"]),
    ))
    return accepted, counters


def contextual_typed_lookup_policy(
    question: str,
    response: str,
    lexical_records: list[dict[str, Any]] | None = None,
    typed_lookup_item: dict[str, Any] | None = None,
    *,
    limit: int = 6,
) -> dict[str, Any]:
    """Return bounded, nonterminal strict evidence for a typed question."""

    request = detect_contextual_lookup_request(question)
    base: dict[str, Any] = {
        "version": VERSION,
        "lookup_mode": "forbidden_for_ordinary_context",
        "applicable": bool(request["applicable"]),
        "operation": request["operation"],
        "source_priority_order": list(SOURCE_PRIORITY_CLASSES),
        "exact_normalized_required": True,
        "operation_key_entity_slot_alignment_required": True,
        "fuzzy_or_semantic_near_match_allowed": False,
        "general_world_rescue_allowed": False,
        "retrieval_miss_is_refutation": False,
        "terminal_label_authority": False,
        "model_nonterminal": True,
        "conflict": False,
        "evidence": [],
    }
    if request["applicable"] is not True:
        base["retrieval_audit"] = {"examined": 0, "accepted": 0}
        return base
    operation = str(request["operation"])
    lexical = _lexical_exact_evidence(operation, question, response, lexical_records)
    retrieved, audit = _strict_retrieval_evidence(
        request, question, response, typed_lookup_item
    )
    combined: list[dict[str, Any]] = []
    seen: set[str] = set()
    for evidence in sorted(
        lexical + retrieved,
        key=lambda value: (
            int(value["source_priority"]),
            0 if value["response_relation"] in ("support_candidate", "counter_candidate") else 1,
            str(value["source_id"]), int(value["source_record_index"]),
        ),
    ):
        identity = str(evidence["evidence_sha256"])
        if identity not in seen:
            seen.add(identity)
            combined.append(evidence)
    # Wikipedia is a last-resort corroboration source.  It is not shown when
    # any aligned book/OCR, curated, or other-authorized exact evidence exists.
    non_wikipedia = [value for value in combined if int(value["source_priority"]) < 3]
    ranked = non_wikipedia if non_wikipedia else combined
    ranked = ranked[:limit]
    answer_sets = {
        tuple(sorted(comparison_view(str(answer)) for answer in value.get("answers", []) if str(answer).strip()))
        for value in ranked if value.get("answers")
    }
    response_roles = {str(value.get("response_relation")) for value in ranked}
    conflict = len(answer_sets) > 1 or {
        "support_candidate", "counter_candidate"
    } <= response_roles
    base.update({
        "lookup_mode": (
            "strict_typed_exact_conflict_nonterminal" if conflict else
            "strict_typed_exact_reference_nonterminal" if ranked else
            "strict_typed_exact_miss_nonterminal"
        ),
        "strict_typed_retrieval_invoked": True,
        "conflict": conflict,
        "evidence": ranked,
        "retrieval_audit": {
            **audit,
            "lexical_exact_accepted": len(lexical),
            "final_bounded_evidence": len(ranked),
            "wikipedia_suppressed_by_higher_priority": bool(non_wikipedia)
            and any(int(value["source_priority"]) == 3 for value in combined),
        },
    })
    return base


def build_contextual_policy_packet(
    context: str,
    question: str,
    response: str,
    router: Callable[..., dict[str, Any]],
    lexical_records: list[dict[str, Any]] | None = None,
    typed_lookup_item: dict[str, Any] | None = None,
    *,
    max_windows: int = 8,
    full_context_char_limit: int = 6000,
    coverage_window_chars: int = 3000,
    coverage_overlap_chars: int = 320,
) -> tuple[str, dict[str, Any]]:
    """Build the v4 full-context packet plus the narrow v5 exception."""

    user, diagnostic = _v4_build_contextual_policy_packet(
        context, question, response, router, lexical_records=[],
        max_windows=max_windows, full_context_char_limit=full_context_char_limit,
        coverage_window_chars=coverage_window_chars,
        coverage_overlap_chars=coverage_overlap_chars,
    )
    policy = contextual_typed_lookup_policy(
        question, response, lexical_records, typed_lookup_item
    )
    if policy["applicable"]:
        user = user.replace(
            '"evidence_universe":"only_the_literal_supplied_context"',
            '"evidence_universe":"literal_context_plus_strict_typed_grammar_reference_only"',
            1,
        ).replace(
            '"external_retrieval_allowed":false',
            '"external_retrieval_allowed":"strict_typed_grammar_reference_only"',
            1,
        ).replace(
            "this is the only evidence universe",
            "primary evidence universe; the exact typed grammar reference below is the sole exception",
            1,
        )
        user += (
            "\n\nSTRICT TYPED GRAMMAR/THEORY REFERENCE (nonterminal):\n"
            "This section is allowed only because the literal question requests the stated operation. "
            "It cannot supply general world facts, cannot override incompatible supplied context, and a "
            "miss or conflict is not refutation. Fuzzy and semantic-near matches were removed.\n"
            + _canonical(policy)
        )
    diagnostic["context_policy_version"] = VERSION
    diagnostic["parent_context_policy_version"] = _V4_VERSION
    diagnostic["operation_axis"] = list(OPERATION_AXIS)
    diagnostic["operation_axis_count"] = len(OPERATION_AXIS)
    diagnostic["typed_lookup_operation_inventory"] = list(TYPED_LOOKUP_OPERATIONS)
    diagnostic["typed_lookup_external_reference_allowed"] = bool(policy["applicable"])
    diagnostic["ordinary_context_external_retrieval_allowed"] = False
    diagnostic["contextual_typed_lookup_policy"] = policy
    # The generated long-context aggregator already passes this field.  Point
    # it at the complete v5 typed policy rather than the v4 three-shell view.
    diagnostic["contextual_lexical_policy"] = policy
    diagnostic["prompt_sha256"] = _sha(user)
    diagnostic.pop("receipt_sha256", None)
    diagnostic["receipt_sha256"] = _sha(_canonical(diagnostic))
    return user, diagnostic


def build_aggregation_user(
    question: str,
    response: str,
    window_results: list[dict[str, Any]],
    *,
    selected_notes: list[dict[str, Any]] | None = None,
    bounded_derivations: list[dict[str, Any]] | None = None,
    lexical_policy: dict[str, Any] | None = None,
) -> str:
    """Keep window passes context-only; admit typed evidence only at final merge."""

    prompt = _v4_build_aggregation_user(
        question, response, window_results,
        selected_notes=selected_notes,
        bounded_derivations=bounded_derivations,
        lexical_policy=lexical_policy,
    )
    if (lexical_policy or {}).get("applicable") is True:
        prompt = prompt.replace(
            "Use no outside knowledge.",
            "Use no general outside knowledge; only the exact typed grammar reference in this packet is admitted.",
            1,
        ).replace(
            "FROZEN EXACT LEXICAL-SHELL POLICY (nonterminal; ordinary retrieval forbidden)",
            "STRICT EXACT TYPED GRAMMAR POLICY (nonterminal; ordinary/general retrieval forbidden)",
            1,
        )
    return prompt


__all__ = [
    "VERSION", "CANONICAL_POLICY_FAMILIES", "ENGINEERED_EVALUATION_CELLS",
    "OPERATION_AXIS", "TYPED_LOOKUP_OPERATIONS", "SOURCE_PRIORITY_CLASSES",
    "PROXY_PREFIX", "comparison_view", "unicode_receipt",
    "detected_policy_families", "full_coverage_windows",
    "build_window_adjudication_user", "detect_contextual_lookup_request",
    "prepare_contextual_typed_lookup_input", "attach_contextual_typed_lookup",
    "contextual_typed_lookup_policy", "build_contextual_policy_packet",
    "build_aggregation_user",
]


In [ ]:
from pipeline.phase2_sparse_retrieval import build as build_sparse_retrieval
from pipeline.phase2_mmap_retrieval import load_mmap_index, close_mmap_index
from pipeline.phase2_composite_fts_retrieval import load as load_fts, retrieve as retrieve_fts
from pipeline.contextual_note_taker_fallback import rank_context_windows

def normalized_text(value: object) -> str:
    return str(value or "").replace("\r\n", "\n").replace("\r", "\n").strip()

rows = []
for source_index, raw in test.iterrows():
    context = normalized_text(raw.context)
    rows.append({
        "example_id": str(raw.id), "source_index": int(source_index),
        "model_prompt_bn": normalized_text(raw.prompt_bn),
        "model_response_bn": normalized_text(raw.response_bn),
        "model_context_bn": context, "context_available": has_context(context),
        "formatting_provenance": {"status": "newline_normalization_only"},
    })

def typed_grammar_operation(prompt: str) -> str:
    folded=" ".join(prompt.casefold().split())
    families=(
        ("antonym",r"বিপরীত(?:ার্থক)?\s*শব্দ"),("sandhi",r"সন্ধি"),("samas",r"সমাস"),
        ("idiom",r"বাগধারা|প্রবাদ"),("prefix_suffix",r"উপসর্গ|প্রত্য[য়য়]"),
        ("one_word_expression",r"এক\s*কথা[য়য়]\s*প্রকাশ"),("word_pair_register",r"শব্দজো[ড়ড়]া|গুরুচণ্ডালী|তৎসম|তদ্ভব|রেজিস্টার"),
        ("natva_satva",r"ণত্ব|নত্ব|ষত্ব"),("spelling_grammar_rule",r"শুদ্ধ\s*বানান|ব্যাকরণ(?:ের)?\s*নিয়ম|ব্যাকরণ(?:ের)?\s*নিয়ম"),
    )
    return next((name for name,pattern in families if re.search(pattern,folded)),"")

closed = [row for row in rows if not row["context_available"]]
typed_context=[]
for row in rows:
    operation=typed_grammar_operation(row["model_prompt_bn"])
    if row["context_available"] and operation:
        proxy=dict(row); proxy["example_id"]="ctxgrammar:"+row["example_id"]
        proxy["context_available"]=False; proxy["typed_context_grammar_operation"]=operation
        typed_context.append(proxy)
retrieval_queries=closed+typed_context
closed_input = WORK_DIR / "morichika_closed_inputs.jsonl"
closed_input.write_text("".join(canonical_json(r)+"\n" for r in retrieval_queries), encoding="utf-8")

# Mandatory source-family health gate: every admitted family is hash-bound,
# openable, and independently search-probed before competition retrieval.
counts = json.loads((MORICHIKA_ROOT / "SOURCE_COUNTS.json").read_text(encoding="utf-8-sig"))
expected_counts = counts["records_by_source"]
health = {"manifest_id": EXPECTED_MANIFEST_ID, "package_id": EXPECTED_PACKAGE_ID,
          "declared_files": len(MORICHIKA_MANIFEST["files"]), "sources": {}, "closed_rows": len(closed),
          "typed_context_exact_grammar_lookup_rows":len(typed_context),"ordinary_context_external_retrieval":False}

mmap_dirs = sorted((MORICHIKA_ROOT / "artifacts/retrieval").glob("*_mmap_v2"))
seen = set()
for directory in mmap_dirs:
    manifest = json.loads((directory / "manifest.json").read_text(encoding="utf-8-sig"))
    sid = str(manifest.get("source_id", ""))
    if sid not in expected_counts or sid in seen: continue
    index = load_mmap_index(directory)
    try:
        probe = None
        for probe_index in range(min(512, len(index["records"]))):
            candidate_probe=index["records"][probe_index]
            if str(candidate_probe.get("question") or "").strip() and (
                candidate_probe.get("answers") or candidate_probe.get("choices") or candidate_probe.get("records")
                or str(candidate_probe.get("supporting_text") or "").strip()
            ):
                probe=candidate_probe; break
        if probe is None:
            raise RuntimeError(f"mmap known-answer record unavailable: {sid}")
        question = str(probe.get("question") or "")
        key = str(probe.get("normalized_question") or "")
        exact = index["exact_lookup"].get(key, [])
        query = index["vectorizer"].transform([question])
        scores = query @ index["matrix"].T
        top_score = float(scores.max()) if scores.nnz else 0.0
        if not question or top_score <= 0:
            raise RuntimeError(f"mmap known-answer probe failed: {sid}")
        health["sources"][sid] = {"kind":"mmap", "path":str(directory), "records":int(manifest["records"]),
                                  "probe_question":question[:120],"probe_answer_count":len(probe.get("answers") or probe.get("records") or []),
                                  "probe_exact_hits":len(exact), "probe_top_score":top_score}
        seen.add(sid)
    finally:
        close_mmap_index(index)
        del index
        gc.collect()

fts_dir = MORICHIKA_ROOT / "artifacts/retrieval/phase2_strict_rights_safe_fts_v3_final"
fts_manifest, connection = load_fts(fts_dir)
try:
    for sid in [str(x[0]) for x in connection.execute("SELECT DISTINCT source_id FROM evidence ORDER BY source_id")]:
        record = connection.execute("SELECT question,title,supporting_text FROM evidence WHERE source_id=? ORDER BY id LIMIT 1", (sid,)).fetchone()
        query = str(record[0] or record[1] or record[2] or "").strip()
        hits = retrieve_fts(connection, query, top_k=8, model_facing_only=False, source_ids=[sid])
        if not query or not any(str(hit.get("source_id")) == sid for hit in hits):
            raise RuntimeError(f"FTS known-answer probe failed: {sid}")
        health["sources"][sid] = {"kind":"fts", "path":str(fts_dir), "records":int(expected_counts[sid]),
                                  "probe_query":query[:120], "probe_hits":len(hits)}
        seen.add(sid)
finally:
    connection.close()

lexical_path = MORICHIKA_ROOT / "artifacts/retrieval/phase2_lexical_seed_v2/lexical_seed_records.jsonl"
lexical_records = [json.loads(line) for line in lexical_path.read_text(encoding="utf-8-sig").splitlines() if line.strip()]
lex_by_source = defaultdict(list)
for record in lexical_records: lex_by_source[str(record["source_id"])].append(record)
for sid, records_for_source in lex_by_source.items():
    probe = records_for_source[0]
    if not str(probe.get("term_key") or "") or not probe.get("accepted_answers"):
        raise RuntimeError(f"lexical known-answer probe failed: {sid}")
    declared_sid="lexical::"+sid
    health["sources"][declared_sid] = {"kind":"lexical_exact", "path":str(lexical_path), "records":len(records_for_source),
                              "probe_term":str(probe["term_key"]), "probe_answers":len(probe["accepted_answers"])}
    seen.add(declared_sid)

missing_sources = sorted(set(expected_counts) - seen)
if missing_sources:
    raise RuntimeError(f"MORICHIKA admitted source families inaccessible: {missing_sources}")
for sid, expected in expected_counts.items():
    if int(health["sources"][sid]["records"]) != int(expected):
        raise RuntimeError(f"source record-count mismatch: {sid}")

retrieval_dir = WORK_DIR / "morichika_closed_retrieval"
retrieval_receipt = build_sparse_retrieval(
    closed_input, retrieval_dir, top_k=8, batch_size=128,
    composite_cache_dir=fts_dir, composite_query_mode="all_closed",
)
retrieved_rows = [json.loads(line) for line in (retrieval_dir/"retrieval.jsonl").read_text(encoding="utf-8-sig").splitlines() if line.strip()]
retrieval_by_id = {str(r["example_id"]): r for r in retrieved_rows}
if len(retrieval_by_id) != len(retrieval_queries):
    raise RuntimeError("closed retrieval output cardinality mismatch")
candidate_rows = sum(bool(retrieval_by_id[r["example_id"]].get("retrieval_candidates")) for r in closed)
if closed and candidate_rows == 0:
    raise RuntimeError("all_closed retrieval returned zero candidates")
health["retrieval_receipt"] = retrieval_receipt
health["closed_rows_with_candidates"] = candidate_rows
health["closed_candidate_coverage"] = candidate_rows / max(1, len(closed))
(WORK_DIR/"morichika_retrieval_health.json").write_text(json.dumps(health, ensure_ascii=False, indent=2), encoding="utf-8")
print("MORICHIKA retrieval health:", len(health["sources"]), "families; closed hits", candidate_rows, "/", len(closed))

def exact_lexical(prompt: str, response: str, limit: int = 3) -> list[dict]:
    p = " ".join(prompt.casefold().split()); r = " ".join(response.casefold().split())
    operation = "antonym_lookup" if "বিপরীত" in p else "idiom_meaning_lookup" if ("বাগধারা" in p or "প্রবাদ" in p) else "prefix_origin_classification" if "উপসর্গ" in p else ""
    if not operation: return []
    prompt_tokens=set(re.findall(r"[A-Za-z0-9\u0980-\u09ff]+",p))
    generic={"অর্থ","মানে","শব্দ","শাব্দিক","বিপরীত","বিপরীতার্থক","বাগধারা","প্রবাদ","উপসর্গ","কি","কী","কোন","কোনটি"}
    found=[]
    for item in lexical_records:
        if item.get("operation") != operation or item.get("conflict_status") != "none": continue
        terms=[str(item.get("term_key") or "")]+[str(v) for v in item.get("display_terms", [])]
        # Generic shell words (for example only ``শাব্দিক অর্থ``) are not an
        # operand.  Require an exact distinctive record-key token in prompt.
        operand_tokens={token for term in terms for token in re.findall(r"[A-Za-z0-9\u0980-\u09ff]+"," ".join(term.casefold().split()))
                        if token not in generic and len(token)>=2}
        if not operand_tokens or not (operand_tokens & prompt_tokens): continue
        accepted=[" ".join(str(v).casefold().split()) for v in item.get("accepted_answers", [])]
        contrast=[" ".join(str(v).casefold().split()) for v in item.get("contrast_answers", [])]
        role="support_candidate" if r in accepted else "counter_candidate" if r in contrast else "neutral_candidate"
        found.append({"source_id":item.get("source_id"),"operation":operation,"term_key":item.get("term_key"),
                      "display_terms":item.get("display_terms",[]),"conflict_status":"none","sense":item.get("sense"),
                      "register":item.get("register"),"etymological_class":item.get("etymological_class"),
                      "accepted_answers":item.get("accepted_answers",[])[:8],"contrast_answers":item.get("contrast_answers",[])[:8],"evidence_role":role})
        if len(found)>=limit: break
    return found

def query_alignment_signature(prompt: str, response: str) -> dict:
    joined=" ".join(prompt.casefold().split())
    tokens=[t for t in re.findall(r"[A-Za-z0-9\u0980-\u09ff]+",joined) if len(t)>=2]
    numbers=re.findall(r"[+-]?[0-9০-৯]+(?:[.,][0-9০-৯]+)?",prompt)
    units=[u for u in ("সাল","বছর","দিন","মাস","কিলোমিটার","মিটার","টাকা","শতাংশ","জন","টি") if u in joined]
    negations=[n for n in ("না","নয়","নয়","নেই","ব্যতীত") if n in tokens]
    relation=[r for r in ("কবে","কোথায়","কোথায়","কে","কার","জন্ম","মৃত্যু","শপথ","অর্থ","বিপরীত","প্রতিষ্ঠাতা","স্রষ্টা","বয়স","বয়স") if r in joined]
    answer_type="date" if any(x in joined for x in ("কবে","তারিখ","কত সালে")) else "person" if re.search(r"(?:কে|কার)\??$",joined) else "location" if ("কোথায়" in joined or "কোথায়" in joined) else "number" if "কত" in joined else "lexical" if any(x in joined for x in ("অর্থ","বিপরীত","বাগধারা","উপসর্গ")) else "text"
    return {"substantive_query_tokens":tokens[:24],"relation_property_cues":relation,"numbers_dates":numbers,"units":units,
            "negation":negations,"answer_type":answer_type,"response_proposition":response[:240]}

def compact_closed_evidence(item: dict, lexical: list[dict], prompt: str, response: str, limit: int = 6) -> tuple[str,list[str],dict]:
    raw_candidates=list(item.get("retrieval_candidates") or [])
    def tier(value, default=99):
        try: return int(value)
        except (TypeError,ValueError): return default
    def alignment_key(candidate):
        # Exact requested relation/slot/answer type precedes authority.  This
        # prevents a wrong-relation passage from starving a lower-ranked exact
        # NCTB/date answer while retaining deterministic authority tie-breaks.
        return (
            candidate.get("model_facing_eligible") is not True,
            tier(candidate.get("semantic_alignment_tier")),
            tier(candidate.get("slot_compatibility_tier")),
            tier(candidate.get("policy_compatibility_tier")),
            candidate.get("number_set_match") is False,
            candidate.get("negation_set_match") is False,
            tier(candidate.get("authority_tier")),
            candidate.get("exact_normalized") is not True,
            str(candidate.get("evidence_role") or "neutral_candidate") == "neutral_candidate",
            tier(candidate.get("rank")),
            str(candidate.get("source_id") or ""),
        )
    signature=query_alignment_signature(prompt,response)
    stop={"কি","কী","কে","কার","কোন","কোনটি","কত","কবে","কোথায়","কোথায়","হয়","হয়","ছিল","আছে","একটি","এই","ও","এবং","সরকার"}
    relation_words=set(signature["relation_property_cues"]+[
        "গ্রহণ","গঠিত","প্রতিষ্ঠিত","অর্থ","বিপরীত","জন্ম","মৃত্যু","বয়স","বয়স","তারিখ","সাল"
    ])
    entity_tokens={t for t in signature["substantive_query_tokens"] if t not in stop and t not in relation_words and len(t)>=3}
    def hard_aligned(candidate):
        text=" ".join([str(candidate.get("question") or ""),str(candidate.get("supporting_text") or "")]).casefold()
        candidate_tokens=set(re.findall(r"[A-Za-z0-9\u0980-\u09ff]+",text))
        entity_ok=not entity_tokens or bool(entity_tokens & candidate_tokens)
        relation_cues={c for c in signature["relation_property_cues"] if c not in {"কবে","কে","কার","কোথায়","কোথায়"}}
        relation_ok=not relation_cues or bool(relation_cues & candidate_tokens)
        return (candidate.get("model_facing_eligible") is True and tier(candidate.get("semantic_alignment_tier"))<=1
                and tier(candidate.get("slot_compatibility_tier"))==0 and tier(candidate.get("policy_compatibility_tier"))<=1
                and candidate.get("number_set_match") is not False and candidate.get("negation_set_match") is not False
                and entity_ok and relation_ok)
    admitted=[candidate for candidate in raw_candidates if hard_aligned(candidate)]
    ranked=sorted(admitted,key=alignment_key)
    # Keep the best aligned candidate first, then preserve source diversity
    # before filling remaining prompt slots by the same alignment order.
    candidates=[]; used_sources=set()
    for candidate in ranked:
        sid=str(candidate.get("source_id") or "")
        if sid not in used_sources:
            candidates.append(candidate); used_sources.add(sid)
        if len(candidates)>=limit: break
    if len(candidates)<limit:
        for candidate in ranked:
            if candidate not in candidates: candidates.append(candidate)
            if len(candidates)>=limit: break
    lines=["QUERY_RESPONSE_ALIGNMENT "+canonical_json(signature)]+["EXACT_LEXICAL "+canonical_json(x) for x in lexical]
    source_ids=[]; roles=Counter()
    for candidate in candidates[:limit]:
        sid=str(candidate.get("source_id") or ""); source_ids.append(sid)
        role=str(candidate.get("evidence_role") or "neutral_candidate"); roles[role]+=1
        lines.append(canonical_json({k:candidate.get(k) for k in (
            "source_id","authority_class","authority_tier","evidence_role","question","answers","choices",
            "supporting_text","exact_normalized","semantic_alignment_tier","slot_compatibility_tier",
            "policy_compatibility_tier","number_set_match","negation_set_match","response_answer_relation")}))
    diagnostics={"query_alignment_signature":signature,"candidate_count":len(raw_candidates),"hard_aligned_candidate_count":len(admitted),
                 "misaligned_candidates_excluded":len(raw_candidates)-len(admitted),"prompt_candidate_count":len(candidates),"source_ids":sorted(set(source_ids)),"evidence_roles":dict(roles),
                 "merged_source_candidate":item.get("merged_source_candidate"),"retrieval_miss_means":"NEI_not_refutation"}
    return "\n".join(lines) if lines else "[NO ALIGNED SOURCE CANDIDATE: treat as NEI, never refutation]", sorted(set(source_ids)), diagnostics

test_aug=test.copy(); test_aug["row_key"]="test_"+test_aug.id.astype(str)
packets=[]
for row in rows:
    if row["context_available"]:
        lexical=exact_lexical(row["model_prompt_bn"],row["model_response_bn"])
        user_packet, route = build_contextual_policy_packet(row["model_context_bn"],row["model_prompt_bn"],row["model_response_bn"],rank_context_windows,
                                                             lexical_records=lexical,max_windows=8,full_context_char_limit=6000)
        operation=typed_grammar_operation(row["model_prompt_bn"])
        grammar_sources=[]; grammar_diag={"lookup_mode":"not_applicable","ordinary_context_external_retrieval":False}
        if operation:
            proxy_item=retrieval_by_id["ctxgrammar:"+row["example_id"]]
            exact_item=dict(proxy_item)
            exact_item["retrieval_candidates"]=[c for c in proxy_item.get("retrieval_candidates",[]) if c.get("exact_normalized") is True]
            grammar_evidence,grammar_sources,grammar_diag=compact_closed_evidence(exact_item,lexical,row["model_prompt_bn"],row["model_response_bn"],limit=4)
            grammar_diag.update({"lookup_mode":"typed_context_exact_grammar_exception","operation":operation,"fuzzy_candidates_exposed":False})
            user_packet += "\n\nTYPED GRAMMAR CLOSED-LOOKUP EXCEPTION (exact operation+operand/key+sense/register only; nonterminal):\n"+grammar_evidence
        packets.append({"morichika_lane":"contextual","phase2_rag_evidence":"","morichika_source_ids":grammar_sources,
                        "morichika_retrieval_diagnostics":grammar_diag,"morichika_context_packet":user_packet,
                        "morichika_context_receipt":route})
    else:
        item=retrieval_by_id[row["example_id"]]; lexical=exact_lexical(row["model_prompt_bn"],row["model_response_bn"])
        evidence,sources,diag=compact_closed_evidence(item,lexical,row["model_prompt_bn"],row["model_response_bn"])
        packets.append({"morichika_lane":"closed_book","phase2_rag_evidence":evidence,"morichika_source_ids":sources,
                        "morichika_retrieval_diagnostics":diag,"morichika_context_packet":"","morichika_context_receipt":{}})
packet_frame=pd.DataFrame(packets)
for col in packet_frame.columns: test_aug[col]=packet_frame[col].tolist()


## Gemma-4-31B Q4 dual-order final-logit judge


In [ ]:
SYSTEM_PROMPT = """You are a strict factual-verification judge fluent in Bengali and English.
Classify whether the candidate Bengali response fully answers the exact question faithfully.

Rules:
- With a supporting passage, treat that passage as the evidence boundary. The answer must express the relation asked by the question; merely appearing somewhere in the passage is not sufficient.
- A SQuAD-BN reference answer is authoritative for its matched question/context. Accept a genuinely equivalent paraphrase, abbreviation, word-form number, or harmless formatting variant.
- Dictionary senses are supporting hints only and can contain multiple senses; select the sense requested by the question.
- Retrieved public passages are non-authoritative hints. Use them only when entity, relation, date, quantity, polarity, and scope match the exact question.
- Without a passage, use factual knowledge and solve elementary arithmetic or logic.
- One material error in person, relation, date, number, place, attribution, meaning, negation, tense, or voice makes the response hallucinated.
- Do not reward fluency and do not invent missing evidence.
- Follow the A/B mapping exactly. Output one letter as the first token."""
MAIN_JUDGE_PROMPT_VERSION = "v5_context_reference_v1"

SEMANTIC_REFERENCE_SYSTEM_PROMPT = """You are a strict bilingual semantic-equivalence verifier.
Decide whether the candidate response has the same requested meaning as the trusted reference answer for the exact current question.
Accept genuine paraphrases, inflection, abbreviations, and word-form numbers. Reject opposites, merely related ideas, different entities or relations, and any changed material detail.
If an exam-bank question is only similar rather than exact, first verify that subject, relation, numbers, and negation transfer; otherwise ignore that bank entry and solve the current question yourself.
Follow the A/B mapping exactly. Output one letter as the first token."""


def compact_context(value: object, max_chars: int = 6000) -> str:
    text = clean_markup(value)
    if max_chars <= 0:
        return "[CONTEXT OMITTED TO FIT MODEL WINDOW]"
    if len(text) <= max_chars:
        return text
    # Preserve both ends; question/response/reference are placed after context
    # and therefore survive tokenizer left-truncation.
    left = max_chars * 2 // 3
    return text[:left] + "\n...[TRUNCATED]...\n" + text[-(max_chars-left):]


def compact_field(value: object, max_chars: int | None) -> str:
    """Preserve both ends of a question/answer only in the last-resort Q4 retry."""
    if max_chars is None:
        # Preserve the exact proven Transformers/Q4 prompt on normal attempts.
        return str(value)
    text = clean_markup(value)
    if len(text) <= max_chars:
        return text
    left = max_chars * 2 // 3
    return text[:left] + " ...[TRUNCATED]... " + text[-(max_chars-left):]


def row_references(row: pd.Series) -> tuple[tuple[str, ...], tuple[str, ...]]:
    trusted_answers = []

    if bool(getattr(row, "squad_found", False)):
        trusted_answers.extend(tuple(row.squad_answers))

    if bool(getattr(row, "phase2_qa_found", False)):
        trusted_answers.extend(tuple(row.phase2_qa_gold_answers))

    unique_trusted = []
    seen = set()

    for answer in trusted_answers:
        text = re.sub(r"\\s+", " ", str(answer)).strip()
        key = canonical_text(text)
        if text and key and key not in seen:
            seen.add(key)
            unique_trusted.append(text)

    dictionary_answers = tuple(row.dictionary_answers)

    return (
        tuple(unique_trusted[:MAX_REFERENCE_ANSWERS]),
        dictionary_answers[:MAX_REFERENCE_ANSWERS],
    )


def build_user_prompt(
    row: pd.Series,
    reverse: bool,
    context_max_chars: int = 6000,
    question_max_chars: int | None = None,
    response_max_chars: int | None = None,
) -> str:
    mapping = "A=HALLUCINATED; B=FAITHFUL" if reverse else "A=FAITHFUL; B=HALLUCINATED"
    context = (
        compact_context(row.context, max_chars=context_max_chars)
        if has_context(row.context)
        else "[NULL]"
    )
    question = compact_field(row.prompt_bn, question_max_chars)
    response = compact_field(row.response_bn, response_max_chars)
    trusted_answers, dictionary_answers = row_references(row)
    reference_lines = []
    if trusted_answers:
        reference_lines.append(
            "Trusted exact-question answer(s): "
            + " | ".join(trusted_answers)
        )
    if dictionary_answers:
        reference_lines.append(
            "Possible dictionary sense(s): "
            + " | ".join(dictionary_answers)
        )
    references = "\n".join(reference_lines) if reference_lines else "[NONE]"
    support_hint = "YES" if bool(row.response_in_context) else "NO"
    retrieved_evidence = (
        compact_context(row.phase2_rag_evidence, max_chars=1800)
        if bool(getattr(row, "phase2_rag_used", False))
        else "[NONE]"
    )
    return f"""Verify the item below.

<supporting_context>
{context}
</supporting_context>
<question>
{question}
</question>
<candidate_response>
{response}
</candidate_response>
<public_reference_answers>
{references}
</public_reference_answers>
<retrieved_public_passages_non_authoritative>
{retrieved_evidence}
</retrieved_public_passages_non_authoritative>
<literal_candidate_in_context>{support_hint}</literal_candidate_in_context>

Output mapping: {mapping}
Verdict:"""


def build_semantic_reference_prompt(row: pd.Series, reverse: bool) -> str:
    """Build the released-sample-validated trusted-reference prompt."""
    mapping = (
        "A=NOT_EQUIVALENT; B=EQUIVALENT"
        if reverse
        else "A=EQUIVALENT; B=NOT_EQUIVALENT"
    )
    return f"""<current_question>{row.prompt_bn}</current_question>
<candidate_response>{row.response_bn}</candidate_response>
<trusted_reference_answer>{row.ling_expected}</trusted_reference_answer>

Does the candidate express the same requested meaning as the reference?
Output mapping: {mapping}
Verdict:"""


def find_gemma_model() -> str:
    if MODEL_PATH_OVERRIDE and Path(MODEL_PATH_OVERRIDE).exists():
        return MODEL_PATH_OVERRIDE
    candidates = []
    if IS_KAGGLE:
        for config_path in INPUT_ROOT.rglob("config.json"):
            try:
                config = json.loads(config_path.read_text(encoding="utf-8"))
            except Exception:
                continue
            if str(config.get("model_type", "")).casefold() != "gemma4":
                continue
            lower = str(config_path.parent).casefold()
            if "assistant" in lower or "base" in lower:
                continue
            candidates.append(config_path.parent)
    if candidates:
        candidates.sort(key=lambda p: ("31b-it" not in str(p).casefold(), len(str(p))))
        return str(candidates[0])
    if ALLOW_ONLINE_MODEL_FALLBACK:
        return MODEL_ID
    raise FileNotFoundError(
        "Attach google/gemma-4/Transformers/gemma-4-31b-it/1, or set "
        "ALLOW_ONLINE_MODEL_FALLBACK=True for a development run."
    )


def prompt_hash(row: pd.Series) -> str:
    squad_answers, dictionary_answers = row_references(row)
    payload = "\u241f".join(
        [
            MAIN_JUDGE_PROMPT_VERSION,
            row.context,
            row.prompt_bn,
            row.response_bn,
            *squad_answers,
            *dictionary_answers,
            str(getattr(row, "phase2_rag_evidence", "")),
        ]
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:20]


def load_transformers_judge():
    import torch
    import transformers
    from transformers import AutoModelForCausalLM, AutoProcessor, BitsAndBytesConfig

    if not torch.cuda.is_available():
        raise RuntimeError("RUN_LLM=True requires a Kaggle GPU accelerator.")
    model_path = find_gemma_model()
    local_only = Path(model_path).exists()
    processor = AutoProcessor.from_pretrained(
        model_path, local_files_only=local_only, trust_remote_code=False
    )
    # Text-only Gemma checkpoints may return TokenizersBackend directly;
    # multimodal processors expose the same tokenizer as `.tokenizer`.
    tokenizer = getattr(processor, "tokenizer", processor)
    tokenizer.padding_side = "left"
    tokenizer.truncation_side = "left"
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    config = json.loads((Path(model_path) / "config.json").read_text(encoding="utf-8")) if local_only else {}
    already_quantized = bool(config.get("quantization_config"))
    kwargs = dict(
        device_map="auto",
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        local_files_only=local_only,
        trust_remote_code=False,
    )
    if not already_quantized:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
    model = AutoModelForCausalLM.from_pretrained(model_path, **kwargs)
    model.eval()
    input_device = model.get_input_embeddings().weight.device
    print("transformers:", transformers.__version__)
    print("model:", model_path)
    print("input device:", input_device)
    # A mutable holder lets cleanup_judge release the final model references
    # before calling empty_cache(). It still unpacks exactly like the old tuple.
    return [torch, processor, tokenizer, model, input_device]


def _find_q4_runtime_wheel(name_fragment: str) -> Path:
    roots = [INPUT_ROOT]
    if not IS_KAGGLE:
        roots.append(Path("kaggle_assets/llama_cpp_cuda"))
    hits: list[Path] = []
    needle = name_fragment.casefold()
    for root in roots:
        if not root.exists():
            continue
        hits.extend(
            path
            for path in root.rglob("*.whl")
            if needle in path.name.casefold()
        )
    if not hits:
        raise FileNotFoundError(
            f"Missing offline wheel containing {name_fragment!r}. Attach "
            "dietorfriedman/gemma4-llama-cpp-cu124-offline."
        )
    hits.sort(key=lambda path: (len(str(path)), str(path)))
    return hits[0]


def install_q4_runtime() -> None:
    """Install the exact CUDA runtime from attached wheels, with no network."""
    import importlib.metadata
    import subprocess

    expected = {"llama-cpp-python": "0.3.33", "diskcache": "5.6.3"}
    installed = {}
    for package in expected:
        try:
            installed[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            installed[package] = None
    # A rerun of the cell can reuse the already imported, pinned build. On a
    # fresh Kaggle process we deliberately install our CUDA wheel even if an
    # unrelated CPU build happens to have the same package version.
    if "llama_cpp" in sys.modules:
        if installed != expected:
            raise RuntimeError(
                "A different llama_cpp build is already imported. Restart the "
                "kernel before running the offline Q4 notebook."
            )
        return

    llama_wheel = _find_q4_runtime_wheel("llama_cpp_python-0.3.33-")
    diskcache_wheel = _find_q4_runtime_wheel("diskcache-5.6.3-")
    print("installing offline Q4 runtime:", llama_wheel.name, diskcache_wheel.name)
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--no-index",
            "--no-deps",
            "--no-cache-dir",
            "--force-reinstall",
            str(llama_wheel),
            str(diskcache_wheel),
        ]
    )
    actual = {
        package: importlib.metadata.version(package)
        for package in expected
    }
    if actual != expected:
        raise RuntimeError(f"Offline Q4 runtime version mismatch: {actual}")


def find_q4_gguf() -> Path:
    override = Path(Q4_MODEL_PATH_OVERRIDE) if Q4_MODEL_PATH_OVERRIDE else None
    if override is not None and override.exists():
        candidate = override
    else:
        hits = []
        for path in INPUT_ROOT.rglob("*.gguf"):
            name = path.name.casefold()
            if (
                path.is_file()
                and "mmproj" not in name
                and "gemma" in name
                and "31b" in name
                and "q4_0" in name
                and ("-it" in name or "_it" in name)
            ):
                hits.append(path)
        if not hits:
            raise FileNotFoundError(
                "Attach google/gemma-4/Gguf/"
                "gemma-4-31b-it-qat-q4_0-gguf/2."
            )
        # The full language-model shard is much larger than ancillary GGUFs.
        candidate = max(hits, key=lambda path: path.stat().st_size)
    if candidate.stat().st_size < 10 * 2**30:
        raise RuntimeError(
            f"Refusing unexpected small GGUF ({candidate.stat().st_size / 2**30:.2f} GiB): "
            f"{candidate}"
        )
    if candidate.stat().st_size >= 50 * 2**30:
        raise RuntimeError(
            f"GGUF exceeds the Phase-2 50-GiB model-weight limit "
            f"({candidate.stat().st_size / 2**30:.2f} GiB): {candidate}"
        )
    return candidate


def require_two_t4_gpus() -> list[tuple[str, int]]:
    import subprocess

    try:
        result = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=name,memory.total",
                "--format=csv,noheader,nounits",
            ],
            check=True,
            capture_output=True,
            text=True,
        )
    except (FileNotFoundError, subprocess.CalledProcessError) as exc:
        raise RuntimeError("The Q4 GGUF backend requires a Kaggle GPU session") from exc
    gpus: list[tuple[str, int]] = []
    for line in result.stdout.splitlines():
        if not line.strip():
            continue
        name, memory = line.rsplit(",", 1)
        gpus.append((name.strip(), int(memory.strip())))
    if len(gpus) != 2 or any("T4" not in name.upper() for name, _ in gpus):
        description = ", ".join(f"{name} ({memory} MiB)" for name, memory in gpus)
        raise RuntimeError(
            "The Phase-2 Q4 notebook requires Kaggle's 2xT4 accelerator; "
            f"detected: {description or 'no NVIDIA GPU'}."
        )
    print("Q4 GPUs:", gpus)
    return gpus


def load_q4_judge():
    gpus = require_two_t4_gpus()
    install_q4_runtime()
    from llama_cpp import Llama, __version__ as llama_cpp_version, llama_cpp

    if llama_cpp_version != "0.3.33":
        raise RuntimeError(f"Expected llama-cpp-python 0.3.33, got {llama_cpp_version}")
    if hasattr(llama_cpp, "llama_supports_gpu_offload"):
        if not bool(llama_cpp.llama_supports_gpu_offload()):
            raise RuntimeError("The attached llama_cpp wheel lacks CUDA GPU offload")

    model_path = find_q4_gguf()
    total_memory = sum(memory for _name, memory in gpus)
    tensor_split = [memory / total_memory for _name, memory in gpus]
    print("GGUF:", model_path)
    print("GGUF GiB:", round(model_path.stat().st_size / 2**30, 3))
    llm = Llama(
        model_path=str(model_path),
        n_ctx=Q4_N_CTX,
        n_batch=Q4_N_BATCH,
        n_ubatch=Q4_N_UBATCH,
        n_threads=4,
        n_threads_batch=8,
        n_gpu_layers=-1,
        split_mode=llama_cpp.LLAMA_SPLIT_MODE_LAYER,
        main_gpu=0,
        tensor_split=tensor_split,
        flash_attn=False,
        offload_kqv=True,
        use_mmap=True,
        use_mlock=False,
        # A final-step logits processor below captures just the A/B pair.
        # Keeping this false avoids retaining a full-vocabulary vector for
        # every prompt token, which made the first benchmark prohibitively
        # slow and memory-heavy.
        logits_all=False,
        seed=SEED,
        verbose=False,
    )

    def q4_token_id(letter: str) -> int:
        ids = llm.tokenize(letter.encode("utf-8"), add_bos=False, special=True)
        if len(ids) != 1:
            raise ValueError(f"Expected one GGUF token for {letter!r}, got {ids}")
        return int(ids[0])

    a_id, b_id = q4_token_id("A"), q4_token_id("B")
    print("llama-cpp-python:", llama_cpp_version)
    print("verdict token ids:", a_id, b_id)
    return {
        "backend": "q4_gguf",
        "llm": llm,
        "a_id": a_id,
        "b_id": b_id,
        "model_path": str(model_path),
    }


def load_judge():
    if MODEL_BACKEND == "q4_gguf":
        return load_q4_judge()
    return load_transformers_judge()


def verdict_token_id(tokenizer, letter: str) -> int:
    ids = tokenizer.encode(letter, add_special_tokens=False)
    if len(ids) != 1:
        raise ValueError(f"Expected one token for {letter!r}, got {ids}")
    return ids[0]


def score_rows_transformers(frame: pd.DataFrame, cache_name: str, judge) -> pd.DataFrame:
    torch, processor, tokenizer, model, input_device = judge
    a_id, b_id = verdict_token_id(tokenizer, "A"), verdict_token_id(tokenizer, "B")
    cache_path = WORK_DIR / cache_name
    cached = pd.read_csv(cache_path) if cache_path.exists() else pd.DataFrame()
    existing = {
        (str(row.row_key), str(row.prompt_hash)): row
        for row in cached.itertuples(index=False)
    } if len(cached) else {}

    records = []
    pending = []
    for idx, row in frame.iterrows():
        row_key = str(row.row_key)
        digest = prompt_hash(row)
        old = existing.get((row_key, digest))
        if old is not None:
            records.append(old._asdict())
        else:
            pending.append((idx, row_key, digest))

    @torch.inference_mode()
    def one_batch(batch_items):
        texts, metadata = [], []
        for idx, row_key, digest in batch_items:
            row = frame.loc[idx]
            for reverse in (False, True):
                chat = processor.apply_chat_template(
                    [
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": build_user_prompt(row, reverse)},
                    ],
                    tokenize=False,
                    add_generation_prompt=True,
                    enable_thinking=False,
                )
                texts.append(chat)
                metadata.append((row_key, digest, reverse))
        encoded = processor(
            text=texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
        ).to(input_device)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            try:
                logits = model(
                    **encoded, use_cache=False, logits_to_keep=1
                ).logits[:, -1, :]
            except TypeError:
                logits = model(**encoded, use_cache=False).logits[:, -1, :]
        ab = logits[:, [a_id, b_id]].float().softmax(dim=-1).cpu().numpy()
        output = []
        for offset in range(0, len(metadata), 2):
            row_key, digest, _ = metadata[offset]
            normal = ab[offset]
            reverse = ab[offset + 1]
            p_normal = float(normal[0])   # normal A = faithful
            p_reverse = float(reverse[1]) # reverse B = faithful
            output.append(
                {
                    "row_key": row_key,
                    "prompt_hash": digest,
                    "p_normal": p_normal,
                    "p_reverse": p_reverse,
                    "p_faithful": (p_normal + p_reverse) / 2.0,
                    "order_gap": abs(p_normal - p_reverse),
                }
            )
        del encoded, logits, ab
        return output

    started = time.time()
    for start in range(0, len(pending), BATCH_ROWS):
        records.extend(one_batch(pending[start : start + BATCH_ROWS]))
        if (start // BATCH_ROWS + 1) % CHECKPOINT_EVERY == 0 or start + BATCH_ROWS >= len(pending):
            pd.DataFrame(records).drop_duplicates(
                ["row_key", "prompt_hash"], keep="last"
            ).to_csv(cache_path, index=False)
            print(f"{cache_name}: {min(start+BATCH_ROWS, len(pending))}/{len(pending)} new rows")

    result = pd.DataFrame(records).drop_duplicates(["row_key", "prompt_hash"], keep="last")
    print(cache_name, "elapsed seconds:", round(time.time() - started, 1))
    return result


def _q4_prompt_hash(row: pd.Series) -> str:
    # Backend/version namespace prevents accidental reuse of the NF4 cache.
    payload = "gemma4_q4_0_callback_v1\u241f" + prompt_hash(row)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:20]


def _is_context_window_error(exc: BaseException) -> bool:
    message = str(exc).casefold()
    return "context window" in message or "requested tokens" in message


def _save_score_checkpoint(records: list[dict], cache_path: Path) -> pd.DataFrame:
    columns = [
        "row_key",
        "prompt_hash",
        "p_normal",
        "p_reverse",
        "p_faithful",
        "order_gap",
        "normal_letter",
        "reverse_letter",
        "q4_context_chars_normal",
        "q4_context_chars_reverse",
    ]
    result = pd.DataFrame(records)
    if result.empty:
        return pd.DataFrame(columns=columns)
    result = result.drop_duplicates(["row_key", "prompt_hash"], keep="last")
    temporary = cache_path.with_name(cache_path.name + ".tmp")
    result.to_csv(temporary, index=False)
    os.replace(temporary, cache_path)
    return result


def _q4_one_order(
    row: pd.Series,
    reverse: bool,
    seed: int,
    judge: dict,
) -> tuple[float, str, int]:
    """Return continuous faithful probability and generated A/B verdict."""
    llm = judge["llm"]
    a_id, b_id = judge["a_id"], judge["b_id"]
    last_window_error: BaseException | None = None
    seen_prompts: set[str] = set()

    attempts = [
        (context_chars, None)
        for context_chars in Q4_CONTEXT_CHAR_FALLBACKS
        if context_chars > 0
    ] + [(0, 800), (0, 400)]
    for context_chars, field_limit in attempts:
        # Final retries also bound anomalously long questions/responses.
        user_prompt = build_user_prompt(
            row,
            reverse,
            context_max_chars=context_chars,
            question_max_chars=field_limit,
            response_max_chars=field_limit,
        )
        if user_prompt in seen_prompts:
            continue
        seen_prompts.add(user_prompt)
        captured: dict[str, np.ndarray] = {}

        def capture_and_mask(_input_ids, logits):
            # llama.cpp invokes this once at the only generation step. Capture
            # raw, pre-mask model scores for calibration, then constrain the
            # emitted token to A or B without altering their relative scores.
            captured["ab"] = np.asarray(
                logits[[a_id, b_id]], dtype=np.float64
            ).copy()
            masked = np.full_like(logits, -np.inf)
            masked[a_id] = logits[a_id]
            masked[b_id] = logits[b_id]
            return masked

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ]
        try:
            result = llm.create_chat_completion(
                messages=messages,
                max_tokens=1,
                temperature=0.0,
                top_p=1.0,
                seed=seed,
                logits_processor=[capture_and_mask],
            )
        except ValueError as exc:
            if not _is_context_window_error(exc):
                raise
            last_window_error = exc
            continue

        choice = result["choices"][0]
        message = choice.get("message", {})
        text = str(message.get("content", choice.get("text", ""))).strip().upper()
        generated = text[:1] if text[:1] in {"A", "B"} else ""
        if "ab" not in captured:
            raise RuntimeError(
                f"Final-step logits processor was not called for {row.row_key}: {result!r}"
            )
        if not generated:
            raise RuntimeError(f"No A/B verdict for {row.row_key}: {result!r}")
        values = np.exp(captured["ab"] - captured["ab"].max())
        p_a = float(values[0] / values.sum())
        p_faithful = (1.0 - p_a) if reverse else p_a
        return p_faithful, generated, context_chars

    raise RuntimeError(
        f"Prompt for {row.row_key} still exceeds Q4_N_CTX={Q4_N_CTX} after "
        "all safe truncation fallbacks"
    ) from last_window_error


def score_rows_q4(frame: pd.DataFrame, cache_name: str, judge: dict) -> pd.DataFrame:
    cache_path = WORK_DIR / cache_name
    try:
        cached = pd.read_csv(cache_path) if cache_path.exists() else pd.DataFrame()
    except (pd.errors.EmptyDataError, pd.errors.ParserError):
        print("ignoring incomplete cache:", cache_path)
        cached = pd.DataFrame()
    required = {
        "row_key", "prompt_hash", "p_normal", "p_reverse", "p_faithful", "order_gap"
    }
    if len(cached) and not required.issubset(cached.columns):
        print("ignoring incompatible cache:", cache_path)
        cached = pd.DataFrame()
    existing = {
        (str(record.row_key), str(record.prompt_hash)): record
        for record in cached.itertuples(index=False)
    } if len(cached) else {}

    records: list[dict] = []
    pending: list[tuple[object, str, str]] = []
    for idx, row in frame.iterrows():
        row_key = str(row.row_key)
        digest = _q4_prompt_hash(row)
        old = existing.get((row_key, digest))
        if old is not None:
            records.append(old._asdict())
        else:
            pending.append((idx, row_key, digest))
    print(f"{cache_name}: resumed {len(records)}, pending {len(pending)}")

    started = time.time()
    fallback_rows = 0
    for position, (idx, row_key, digest) in enumerate(pending, start=1):
        row = frame.loc[idx]
        stable_seed = (SEED + int(digest[:8], 16)) % (2**31 - 1)
        p_normal, normal_letter, normal_context = _q4_one_order(
            row, False, stable_seed, judge
        )
        p_reverse, reverse_letter, reverse_context = _q4_one_order(
            row, True, (stable_seed + 17) % (2**31 - 1), judge
        )
        fallback_rows += int(
            normal_context != Q4_CONTEXT_CHAR_FALLBACKS[0]
            or reverse_context != Q4_CONTEXT_CHAR_FALLBACKS[0]
        )
        records.append(
            {
                "row_key": row_key,
                "prompt_hash": digest,
                "p_normal": p_normal,
                "p_reverse": p_reverse,
                "p_faithful": (p_normal + p_reverse) / 2.0,
                "order_gap": abs(p_normal - p_reverse),
                "normal_letter": normal_letter,
                "reverse_letter": reverse_letter,
                "q4_context_chars_normal": normal_context,
                "q4_context_chars_reverse": reverse_context,
            }
        )
        if position % CHECKPOINT_EVERY == 0 or position == len(pending):
            _save_score_checkpoint(records, cache_path)
            print(
                f"{cache_name}: {position}/{len(pending)} new rows; "
                f"context fallbacks={fallback_rows}"
            )

    if pending:
        result = _save_score_checkpoint(records, cache_path)
    elif records:
        result = pd.DataFrame(records).drop_duplicates(
            ["row_key", "prompt_hash"], keep="last"
        )
    else:
        result = _save_score_checkpoint([], cache_path)
    print(cache_name, "elapsed seconds:", round(time.time() - started, 1))
    return result


def score_rows(frame: pd.DataFrame, cache_name: str, judge) -> pd.DataFrame:
    if MODEL_BACKEND == "q4_gguf":
        return score_rows_q4(frame, cache_name, judge)
    return score_rows_transformers(frame, cache_name, judge)


SEMANTIC_REFERENCE_SCORE_COLUMNS = [
    "row_key",
    "semantic_reference_hash",
    "semantic_p_normal",
    "semantic_p_reverse",
    "p_semantic_equivalent",
    "semantic_reference_order_gap",
    "semantic_reference_normal_label",
    "semantic_reference_reverse_label",
    "semantic_reference_agree",
    "semantic_reference_label",
]


def _semantic_reference_prompt_hash(row: pd.Series) -> str:
    payload = "\u241f".join(
        [
            "gemma4_q4_semantic_reference_v1",
            str(row.prompt_bn),
            str(row.response_bn),
            str(row.ling_expected),
        ]
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:20]


def _q4_semantic_reference_one_order(
    row: pd.Series,
    reverse: bool,
    seed: int,
    judge: dict,
) -> tuple[float, str]:
    """Return the probability of semantic equivalence for one label order."""
    llm = judge["llm"]
    a_id, b_id = judge["a_id"], judge["b_id"]
    captured: dict[str, np.ndarray] = {}

    def capture_and_mask(_input_ids, logits):
        captured["ab"] = np.asarray(logits[[a_id, b_id]], dtype=np.float64).copy()
        masked = np.full_like(logits, -np.inf)
        masked[a_id] = logits[a_id]
        masked[b_id] = logits[b_id]
        return masked

    result = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": SEMANTIC_REFERENCE_SYSTEM_PROMPT},
            {
                "role": "user",
                "content": build_semantic_reference_prompt(row, reverse),
            },
        ],
        max_tokens=1,
        temperature=0.0,
        top_p=1.0,
        seed=seed,
        logits_processor=[capture_and_mask],
    )
    choice = result["choices"][0]
    message = choice.get("message", {})
    text = str(message.get("content", choice.get("text", ""))).strip().upper()
    generated = text[:1] if text[:1] in {"A", "B"} else ""
    if "ab" not in captured:
        raise RuntimeError(
            f"Semantic-reference logits processor was not called for {row.row_key}"
        )
    if not generated:
        raise RuntimeError(
            f"No semantic-reference A/B verdict for {row.row_key}: {result!r}"
        )
    values = np.exp(captured["ab"] - captured["ab"].max())
    p_a = float(values[0] / values.sum())
    p_equivalent = (1.0 - p_a) if reverse else p_a
    return p_equivalent, generated


def _save_semantic_reference_checkpoint(
    records: list[dict], cache_path: Path
) -> pd.DataFrame:
    if not records:
        return pd.DataFrame(columns=SEMANTIC_REFERENCE_SCORE_COLUMNS)
    result = pd.DataFrame(records).drop_duplicates(
        ["row_key", "semantic_reference_hash"], keep="last"
    )
    result = result[SEMANTIC_REFERENCE_SCORE_COLUMNS]
    temporary = cache_path.with_name(cache_path.name + ".tmp")
    result.to_csv(temporary, index=False)
    os.replace(temporary, cache_path)
    return result


def score_semantic_reference_rows(
    frame: pd.DataFrame,
    cache_name: str,
    judge: dict,
) -> pd.DataFrame:
    """Score trusted linguistic references and abstain on order disagreement.

    This deliberately remains Q4-only: the 20/20 released-sample agreement
    audit was performed on the exact official Gemma-4 Q4_0 artifact.
    """
    if MODEL_BACKEND != "q4_gguf":
        return pd.DataFrame(columns=SEMANTIC_REFERENCE_SCORE_COLUMNS)
    cache_path = WORK_DIR / cache_name
    try:
        cached = pd.read_csv(cache_path) if cache_path.exists() else pd.DataFrame()
    except (pd.errors.EmptyDataError, pd.errors.ParserError):
        print("ignoring incomplete semantic-reference cache:", cache_path)
        cached = pd.DataFrame()
    required = set(SEMANTIC_REFERENCE_SCORE_COLUMNS)
    if len(cached) and not required.issubset(cached.columns):
        print("ignoring incompatible semantic-reference cache:", cache_path)
        cached = pd.DataFrame()
    existing = {
        (str(record.row_key), str(record.semantic_reference_hash)): record
        for record in cached.itertuples(index=False)
    } if len(cached) else {}

    records: list[dict] = []
    pending: list[tuple[object, str, str]] = []
    for idx, row in frame.iterrows():
        row_key = str(row.row_key)
        digest = _semantic_reference_prompt_hash(row)
        old = existing.get((row_key, digest))
        if old is not None:
            records.append(old._asdict())
        else:
            pending.append((idx, row_key, digest))
    print(f"{cache_name}: resumed {len(records)}, pending {len(pending)}")

    started = time.time()
    for position, (idx, row_key, digest) in enumerate(pending, start=1):
        row = frame.loc[idx]
        stable_seed = (SEED + int(digest[:8], 16)) % (2**31 - 1)
        p_normal, normal_letter = _q4_semantic_reference_one_order(
            row, False, stable_seed, judge
        )
        p_reverse, reverse_letter = _q4_semantic_reference_one_order(
            row, True, (stable_seed + 17) % (2**31 - 1), judge
        )
        normal_label = int(p_normal >= 0.5)
        reverse_label = int(p_reverse >= 0.5)
        agrees = normal_label == reverse_label
        records.append(
            {
                "row_key": row_key,
                "semantic_reference_hash": digest,
                "semantic_p_normal": p_normal,
                "semantic_p_reverse": p_reverse,
                "p_semantic_equivalent": (p_normal + p_reverse) / 2.0,
                "semantic_reference_order_gap": abs(p_normal - p_reverse),
                "semantic_reference_normal_label": normal_label,
                "semantic_reference_reverse_label": reverse_label,
                "semantic_reference_agree": agrees,
                "semantic_reference_label": normal_label if agrees else np.nan,
            }
        )
        if position % CHECKPOINT_EVERY == 0 or position == len(pending):
            _save_semantic_reference_checkpoint(records, cache_path)
            print(
                f"{cache_name}: {position}/{len(pending)} new rows; "
                f"agreed={sum(bool(record['semantic_reference_agree']) for record in records)}"
            )
    result = (
        _save_semantic_reference_checkpoint(records, cache_path)
        if pending
        else pd.DataFrame(records, columns=SEMANTIC_REFERENCE_SCORE_COLUMNS)
    )
    print(cache_name, "elapsed seconds:", round(time.time() - started, 1))
    return result


def cleanup_judge(judge) -> None:
    if MODEL_BACKEND == "q4_gguf":
        llm = judge.pop("llm", None)
        if llm is not None:
            close = getattr(llm, "close", None)
            if callable(close):
                close()
            del llm
        judge.clear()
        gc.collect()
        return
    torch, processor, tokenizer, model, input_device = judge
    judge.clear()
    del model, processor, tokenizer, input_device
    gc.collect()
    torch.cuda.empty_cache()


def seed_saved_cache(cache_name: str) -> None:
    """Resume only local working checkpoints; never import attached scores."""
    if not ALLOW_INPUT_SCORE_CACHE:
        return
    target = WORK_DIR / cache_name
    if target.exists():
        return
    aliases = {
        "gemma4_31b_nf4_sample_scores.csv": ("gemma4_31b_nf4_sample_scores.csv",),
        "gemma4_31b_nf4_test_scores.csv": ("gemma4_31b_nf4_test_scores.csv",),
        "gemma4_31b_q4_0_gguf_sample_scores.csv": (
            "gemma4_31b_q4_0_gguf_sample_scores.csv",
        ),
        "gemma4_31b_q4_0_gguf_test_scores.csv": (
            "gemma4_31b_q4_0_gguf_test_scores.csv",
        ),
    }.get(cache_name, (cache_name,))
    roots = [INPUT_ROOT]
    if not IS_KAGGLE:
        roots.append(Path("."))
    for root in roots:
        for alias in aliases:
            hits = sorted(root.rglob(alias))
            for hit in hits:
                if hit.resolve() != target.resolve():
                    shutil.copy2(hit, target)
                    print("seeded cache:", target, "from", hit)
                    return
        for archive in sorted(root.rglob("*.zip")):
            try:
                with zipfile.ZipFile(archive) as zf:
                    names = {Path(name).name: name for name in zf.namelist()}
                    for alias in aliases:
                        if alias in names:
                            target.write_bytes(zf.read(names[alias]))
                            print("seeded cache:", target, "from", archive)
                            return
            except zipfile.BadZipFile:
                continue


DELIB_SYSTEM_PROMPT = """You answer Bengali factual and examination questions accurately.
The candidate answer is deliberately hidden. Independently return only the shortest correct
answer or calculated result. Use textbook evidence only when it answers the same relation.
Do not explain and do not add a verdict."""


def build_deliberation_prompt(row: pd.Series) -> str:
    nctb_lines = "[NONE]"
    if bool(row.nctb_safe_reference):
        nctb_lines = (
            f"Matched textbook question: {row.nctb_source_question}\n"
            f"Textbook answer(s): {' | '.join(tuple(row.nctb_answers)[:MAX_REFERENCE_ANSWERS])}"
        )
        if str(row.nctb_rationale).strip():
            nctb_lines += f"\nTextbook rationale: {str(row.nctb_rationale)[:1000]}"
    return f"""Independently answer this closed-book Bengali item.

<question>{row.prompt_bn}</question>
<licensed_textbook_evidence>{nctb_lines}</licensed_textbook_evidence>

Return only the shortest final answer. Do not explain."""


def parse_deliberation(text: object) -> tuple[float, bool]:
    value = str(text).strip()
    matches = re.findall(r"FINAL\s*[:：=-]\s*([01])", value, flags=re.IGNORECASE)
    if matches:
        label = int(matches[-1])
        return (0.94 if label else 0.06), True
    tail = value[-300:].casefold()
    if "hallucinated" in tail or "ভুল" in tail or "অসত্য" in tail:
        return 0.08, True
    if "faithful" in tail or "সঠিক" in tail or "নির্ভুল" in tail:
        return 0.92, True
    return 0.5, False


def build_deliberation_verdict_prompt(row: pd.Series, reasoning: str, reverse: bool) -> str:
    mapping = "A=HALLUCINATED; B=FAITHFUL" if reverse else "A=FAITHFUL; B=HALLUCINATED"
    return f"""Use the independent scratch analysis to issue the final verdict.
Question: {row.prompt_bn}
Candidate: {row.response_bn}
Scratch analysis:
{reasoning[-2200:]}

Check the relation and calculation yourself; do not blindly copy a verdict stated in scratch.
Output mapping: {mapping}
Output exactly one letter. Verdict:"""


def deliberate_rows(frame: pd.DataFrame, cache_name: str, judge) -> pd.DataFrame:
    if MODEL_BACKEND != "transformers":
        raise RuntimeError("Deliberation is disabled for the Q4 GGUF backend")
    torch, processor, tokenizer, model, input_device = judge
    cache_path = WORK_DIR / cache_name
    cached = pd.read_csv(cache_path) if cache_path.exists() else pd.DataFrame()
    existing = {
        (str(row.row_key), str(row.delib_hash)): row
        for row in cached.itertuples(index=False)
    } if len(cached) else {}
    records, pending = [], []
    for idx, row in frame.iterrows():
        payload = "\u241f".join(
            [row.prompt_bn, row.response_bn, str(row.nctb_source_question), str(row.nctb_answers)]
        )
        digest = hashlib.sha256(payload.encode("utf-8")).hexdigest()[:20]
        old = existing.get((str(row.row_key), digest))
        if old is not None:
            records.append(old._asdict())
        else:
            pending.append((idx, str(row.row_key), digest))

    @torch.inference_mode()
    def one_batch(items):
        texts = []
        for idx, _row_key, _digest in items:
            row = frame.loc[idx]
            texts.append(
                processor.apply_chat_template(
                    [
                        {"role": "system", "content": DELIB_SYSTEM_PROMPT},
                        {"role": "user", "content": build_deliberation_prompt(row)},
                    ],
                    tokenize=False,
                    add_generation_prompt=True,
                        enable_thinking=False,
                )
            )
        encoded = processor(
            text=texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
        ).to(input_device)
        prompt_width = encoded["input_ids"].shape[1]
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            generated = model.generate(
                **encoded,
                max_new_tokens=MAX_DELIB_TOKENS,
                do_sample=False,
                use_cache=True,
                pad_token_id=tokenizer.pad_token_id,
            )
        decoded = processor.batch_decode(
            generated[:, prompt_width:], skip_special_tokens=True
        )
        a_id, b_id = verdict_token_id(tokenizer, "A"), verdict_token_id(tokenizer, "B")
        verdict_texts = []
        for (idx, _row_key, _digest), reasoning in zip(items, decoded):
            row = frame.loc[idx]
            for reverse in (False, True):
                verdict_texts.append(
                    processor.apply_chat_template(
                        [
                            {
                                "role": "system",
                                "content": "You are a strict Bengali factual-verification judge. Follow the A/B mapping and output one letter.",
                            },
                            {
                                "role": "user",
                                "content": build_deliberation_verdict_prompt(row, reasoning, reverse),
                            },
                        ],
                        tokenize=False,
                        add_generation_prompt=True,
                        enable_thinking=False,
                    )
                )
        verdict_encoded = processor(
            text=verdict_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
        ).to(input_device)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            try:
                verdict_logits = model(
                    **verdict_encoded, use_cache=False, logits_to_keep=1
                ).logits[:, -1, :]
            except TypeError:
                verdict_logits = model(**verdict_encoded, use_cache=False).logits[:, -1, :]
        ab = verdict_logits[:, [a_id, b_id]].float().softmax(dim=-1).cpu().numpy()
        output = []
        for offset, ((_idx, row_key, digest), reasoning) in enumerate(zip(items, decoded)):
            normal = ab[2 * offset]
            reverse = ab[2 * offset + 1]
            p_normal = float(normal[0])
            p_reverse = float(reverse[1])
            parsed_probability, parsed_valid = parse_deliberation(reasoning)
            output.append(
                {
                    "row_key": row_key,
                    "delib_hash": digest,
                    "p_deliberation": (p_normal + p_reverse) / 2.0,
                    "delib_order_gap": abs(p_normal - p_reverse),
                    "delib_valid": 1,
                    "parsed_delib_probability": parsed_probability,
                    "parsed_delib_valid": int(parsed_valid),
                    "deliberation": reasoning,
                }
            )
        del encoded, generated, verdict_encoded, verdict_logits, ab
        return output

    started = time.time()
    for start in range(0, len(pending), DELIB_BATCH_ROWS):
        records.extend(one_batch(pending[start : start + DELIB_BATCH_ROWS]))
        if (start // DELIB_BATCH_ROWS + 1) % CHECKPOINT_EVERY == 0 or start + DELIB_BATCH_ROWS >= len(pending):
            pd.DataFrame(records).drop_duplicates(
                ["row_key", "delib_hash"], keep="last"
            ).to_csv(cache_path, index=False)
            print(f"{cache_name}: {min(start+DELIB_BATCH_ROWS, len(pending))}/{len(pending)} new rows")
    result = pd.DataFrame(records).drop_duplicates(["row_key", "delib_hash"], keep="last")
    print(cache_name, "elapsed seconds:", round(time.time() - started, 1))
    return result


def route_fastcheck_rows(frame: pd.DataFrame, scores: pd.DataFrame, limit: int) -> pd.DataFrame:
    """Select a label-free, high-value subset for the short independent answer pass."""
    candidates = frame[~frame.has_context.astype(bool)].copy()
    score_columns = scores[["row_key", "p_faithful", "order_gap"]].drop_duplicates("row_key")
    candidates = candidates.merge(score_columns, on="row_key", how="left")
    prompt_key = candidates.prompt_key.astype(str)
    factual_marker = prompt_key.str.contains(
        "কে|কার|কখন|কবে|কত সালে|কোন সালে|কোথায়|কোথায়|জন্ম|মৃত্যু|রচয়িতা|লেখক|স্থপতি|প্রতিষ্ঠিত",
        regex=True,
    )
    candidates["fastcheck_priority"] = (
        20.0 * candidates.nctb_safe_reference.astype(float)
        + 4.0 * candidates.nctb_question_similarity.clip(0, 1)
        + 3.0 * candidates.response_has_number.astype(float)
        + 2.0 * factual_marker.astype(float)
        + 1.5 * (candidates.response_tokens <= 8).astype(float)
        + 2.0 * candidates.order_gap.fillna(0).clip(0, 1)
        + 0.5 * (1.0 - (candidates.p_faithful.fillna(0.5) - 0.5).abs() * 2.0)
    )
    selected = candidates.sort_values(
        ["fastcheck_priority", "row_key"], ascending=[False, True]
    ).head(min(limit, len(candidates)))
    print(
        "fast-check selected:", len(selected), "/", len(candidates),
        "priority range:",
        round(float(selected.fastcheck_priority.min()), 3) if len(selected) else None,
        "to", round(float(selected.fastcheck_priority.max()), 3) if len(selected) else None,
    )
    # Remove merge-only score columns; the original frame schema is sufficient.
    return selected.drop(columns=["p_faithful", "order_gap", "fastcheck_priority"])

In [ ]:
SYSTEM_PROMPT = """You are MORICHIKA, a strict Bengali/English factuality verifier. Follow lane isolation and the frozen precedence exactly.
CONTEXTUAL lane: supplied context is the ordinary evidence boundary; never use factual/world retrieval. One narrow exception is recognized typed Bengali-language lexical or THEORY/RULE genres: antonym, idiom/bagdhara, prefix/suffix, sandhi, samas, one-word expression, word-pair/register/guruchandali, natva/satva, spelling and grammar rules may consult only exact canonical operation+operand/key+sense/register evidence. That lookup is nonterminal and must be reconciled with supplied context and response; fuzzy or generic factual rescue remains forbidden. Check premise/answerability first, then exact question slot and answer type, entity-relation-property, event phase, date/time/role/place, number/unit, negation/quantifier/comparator/modality/clause scope. Direct support is not mandatory: bounded calculation, age/duration/relative timeline, kinship, definition/formula/theory/rule application are allowed only from supplied operands. Unicode NFC/Bengali-vs-ASCII digits/joiner/attested OCR word break can be equivalent, but near-looking different words are not. Grammar requires the exact operation; antonym/idiom/prefix exact pair/sense/register and guruchandali matter; samas/sandhi/affix/natva/satva require exact rule and operands. Partial containment cannot hide an extra false claim. Missing/ambiguous/conflicting evidence is NOT_ENOUGH_INFORMATION, distinct from directly REFUTED.
CLOSED-BOOK lane: use MORICHIKA offline evidence as nonterminal evidence. Alignment precedes authority: exact operation, slot, entity, relation/property, answer type, date/number/unit, negation, cultural/as-of scope; then books/user OCR/BCS, curated datasets, then other corroboration. Consider support and counterevidence. A fuzzy hit is not proof; retrieval miss means NEI, never refutation. Lexical evidence is valid only for an exact recognized shell/key/sense/register with no conflict.
The frozen context inventory contains 17 canonical policy families, 26 engineered evaluation cells, and 15 operation axes. One material mismatch makes the response hallucinated. Do not reward fluency. Follow the A/B mapping exactly and emit one letter as the first token."""
MAIN_JUDGE_PROMPT_VERSION = "morichika-final31b-strict-v3-17x26x15"

def build_user_prompt(row: pd.Series, reverse: bool, context_max_chars: int=6000,
                      question_max_chars: int|None=None, response_max_chars: int|None=None) -> str:
    mapping="A=HALLUCINATED; B=FAITHFUL" if reverse else "A=FAITHFUL; B=HALLUCINATED"
    question=compact_field(row.prompt_bn,question_max_chars); response=compact_field(row.response_bn,response_max_chars)
    if row.morichika_lane == "contextual":
        packet=str(row.morichika_context_packet)
        # Context policy runtime already performs question-conditioned, source-linked window selection.
        if context_max_chars < 6000:
            packet=compact_context(packet,max_chars=max(400,context_max_chars))
        evidence="[EXTERNAL RETRIEVAL FORBIDDEN]"
    else:
        packet="[NO SUPPLIED CONTEXT]"
        evidence=compact_context(row.phase2_rag_evidence,max_chars=min(2400,max(800,context_max_chars)))
    lane_semantics=("CONTEXTUAL: faithful requires support/valid derivation from supplied context or the narrow exact typed-grammar exception. NEI differs from refutation diagnostically but is not faithful support."
                    if row.morichika_lane=="contextual" else
                    "CLOSED-BOOK: a retrieval miss/NEI is not evidence that the candidate is false; use model knowledge plus admitted nonterminal evidence and counterevidence.")
    return f"""<lane>{row.morichika_lane}</lane>
<lane_label_semantics>{lane_semantics}</lane_label_semantics>
<morichika_top_aligned_evidence_first>{evidence}</morichika_top_aligned_evidence_first>
<context_policy_packet>{packet}</context_policy_packet>
<exact_question>{question}</exact_question>
<candidate_response>{response}</candidate_response>
Decide whether the response is fully faithful under the lane-specific semantics. Mapping: {mapping}\nVerdict:"""

def _q4_prompt_hash(row: pd.Series) -> str:
    payload="\u241f".join([MAIN_JUDGE_PROMPT_VERSION,SYSTEM_PROMPT,str(row.morichika_lane),str(row.context),str(row.prompt_bn),str(row.response_bn),
                            str(row.phase2_rag_evidence),canonical_json(row.morichika_retrieval_diagnostics)])
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:20]


In [ ]:
test_cache_name="morichika_gemma31b_full_corpus_v5_scores.csv"
judge=load_judge()
scores=score_rows(test_aug,test_cache_name,judge)
cleanup_judge(judge); del judge; gc.collect()
if len(scores)!=len(test_aug) or scores.row_key.astype(str).duplicated().any():
    raise RuntimeError("Gemma score cardinality failure; refusing fallback labels")
test_meta=test_aug.merge(scores[["row_key","prompt_hash","p_normal","p_reverse","p_faithful","order_gap","normal_letter","reverse_letter"]],on="row_key",how="left",validate="one_to_one")
if test_meta[["p_normal","p_reverse","p_faithful"]].isna().any().any():
    raise RuntimeError("missing model score; refusing silent default")
test_meta["final_label"]=(test_meta.p_faithful>=0.5).astype(int)
submission=test_meta[["id"]].copy(); submission["label"]=test_meta.final_label.astype(int)
if submission.id.tolist()!=sample_submission.id.tolist() or not submission.label.isin([0,1]).all():
    raise RuntimeError("submission contract failed")
submission.to_csv(WORK_DIR/"submission.csv",index=False)

diagnostics=test_meta[["id","morichika_lane","context","prompt_bn","response_bn","phase2_rag_evidence","morichika_source_ids",
                       "morichika_retrieval_diagnostics","morichika_context_receipt","p_normal","p_reverse","p_faithful","order_gap","final_label"]].copy()
for col in ("morichika_source_ids","morichika_retrieval_diagnostics","morichika_context_receipt"):
    diagnostics[col]=diagnostics[col].map(lambda x: canonical_json(x))
diagnostics.to_csv(WORK_DIR/"morichika_per_row_evidence_diagnostics.csv",index=False)
report={"version":"morichika-final31b-strict-v3","test_rows":len(test_meta),"context_rows":int((test_meta.morichika_lane=="contextual").sum()),
        "closed_rows":int((test_meta.morichika_lane=="closed_book").sum()),"manifest_id":EXPECTED_MANIFEST_ID,"package_id":EXPECTED_PACKAGE_ID,
        "source_family_count":len(health["sources"]),"closed_rows_with_candidates":candidate_rows,"model":Q4_MODEL_ID,
        "dual_order":True,"threshold":0.5,"fallback_labels_used":0,"submission_sha256":file_sha256(WORK_DIR/"submission.csv")}
(WORK_DIR/"morichika_run_receipt.json").write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding="utf-8")
print(json.dumps(report,ensure_ascii=False,indent=2)); print(submission.label.value_counts().sort_index())
